# From messy files to AI-ready data: a complete, beginner-friendly walkthrough

**FreshCart retail demo · Landing → Bronze → Silver → Gold → Semantic**

This notebook rebuilds the whole FreshCart data warehouse **from zero**, one table at a time, and explains
every transformation as it happens: what the table is, what is wrong with the data coming in, what we do to it,
and **why**.

It assumes you know **nothing about data engineering**. Part A teaches the theory you need (with tiny runnable
examples). Parts B to H then apply that theory to real (synthetic) retail data.

| Part | What you learn |
|---|---|
| **A · Theory** | What data engineering is, tables, grain, keys, the medallion architecture, star schemas, measures, a SQL crash course, retail vocabulary |
| **B · Setup** | Connect to an empty local warehouse |
| **C · The business questions** | Why we build any of this |
| **D · Landing zone** | The raw files, exactly as the source systems send them |
| **E · Bronze** (15 tables + 1 log) | Land every file unchanged, with audit columns, incrementally |
| **F · Silver** (13 tables) | Type, clean, decode, deduplicate, conform, protect personal data, quarantine bad rows |
| **G · Gold** (7 tables) | Build the business star schema: 5 dimensions and 2 facts |
| **H · Semantic layer** | Define every KPI once so nobody (and no AI) calculates it wrongly |
| **I · Proving it** | Data-quality rules, an independent raw-to-gold reconciliation, one receipt traced through every layer |
| **J · Cheat sheet** | Every table on one page, the local ↔ Databricks mapping, and a glossary |

**How to run it:** from the repository root, `pip install -r notebooks/requirements.txt`, then
`jupyter notebook notebooks/FreshCart_Data_Transformation_Walkthrough.ipynb` and run the cells **top to bottom**
(Kernel → Restart & Run All). The whole thing takes about a minute. Every number you see below is produced by
the cells, not typed by hand.

> The SQL that transforms each table is **not copied** into this notebook: each step prints and runs the real file
> from `pipeline/silver/` or `pipeline/gold/`, so this notebook can never drift from the project's pipeline.
> Around each file we run small extra queries that show one transformation at a time.

---
# Part A · The theory you need first

## A1 · What is data engineering, and why does AI need it?

A company's data lives in many **source systems**: the tills in the stores (POS), the website, the ERP
(product and cost master data), the CRM (loyalty members), Finance's spreadsheets. Each system was built for its
own job, not for analysis, so each one:

* names the same thing differently (store `01042` in the tills, `1042` in the ERP),
* stores dates, numbers and codes in its own format (`20260914`, `14/09/2026`, `09/14/2026`),
* contains mistakes, duplicates, test data and things that are not what they look like (a "void" line on a receipt is not a sale).

**Data engineering** is the work of moving that data into one place and turning it into tables that are
**correct, consistent, documented and easy to query**. The result is usually called a *data warehouse* or *lakehouse*.

Why does this matter for AI? Tools like **Databricks Genie** let a business user ask a question in English
("what were net sales by region last week?"), and an AI model writes the SQL. The model can only be as right as the
data it sees. If "sales" could mean three columns, if "last week" needs a 4-5-4 fiscal calendar the model has never
seen, or if a join silently doubles every row, the AI will produce a confident, wrong answer.
**AI-ready data** means the hard decisions are made once, in the pipeline, and written down, so the AI only has to
pick the right table and filter.

## A2 · Tables, rows, columns and data types

A **table** is a grid. Each **row** is one thing (one receipt line, one store, one day). Each **column** is one
attribute of that thing, with a **data type**:

| Type | Examples | Why the type matters |
|---|---|---|
| TEXT / STRING | `'FreshCart Boston Seaport'`, `'01042'` | `'01042'` and `'1042'` are different strings: joins fail |
| INTEGER | `2026`, `34` | you can compare and add them |
| REAL / DECIMAL | `3.49`, `1.254` | money and weights; `'3.49'` as text can't be summed reliably |
| DATE | `2026-09-14` | you can do "the day before", "which fiscal week" |
| BOOLEAN | `TRUE`/`FALSE` (1/0 in SQLite) | flags such as `is_promo_sale` |

A table's list of columns and types is its **schema**.

## A3 · Grain: the single most important word

The **grain** of a table is the answer to *"what does one row represent?"*. For example:

* `silver.pos_sales_line`: **one row per line on a till receipt**
* `gold.dim_store`: **one row per store**
* `gold.fct_inventory_daily`: **one row per store, per product, per day**

If you don't know a table's grain you can't use it safely. Joining a "one row per product" table to a table that
actually has **two** rows per product silently doubles your sales (you will see this exact trap in Silver 04).
Every step in this notebook starts by stating its grain.

## A4 · Keys

| Term | Meaning | Example here |
|---|---|---|
| **Natural key** | The id the source system uses | receipt `88120431`, line `2`, store `01042` |
| **Primary key (PK)** | Column(s) whose value is unique for every row. Enforces the grain | `dim_store.store_id` |
| **Composite key** | A primary key made of several columns | `(snapshot_date, store_id, product_id)` |
| **Foreign key (FK)** | A column that points at another table's primary key | `fct_sales_line.store_id → dim_store.store_id` |
| **Pseudonymous key** | An id derived from personal data so it can be joined but not read | `customer_id = sha2(card_number)` |

## A5 · ETL / ELT, batch and incremental, idempotent

* **ETL / ELT** = Extract, Transform, Load. Modern lakehouses do **ELT**: load the raw data first, transform it
  afterwards with SQL inside the warehouse. That's what we do.
* **Batch** = process data in chunks (every night). **Incremental** = on each run, only process what is **new**
  since the last run (new files, rows newer than a *watermark*).
* **Idempotent** = running the same step twice gives the same result as running it once. No duplicates, no drift.
  This is what lets you safely re-run a pipeline after a failure. We prove it in Part E and Part G.
* **Upsert / MERGE** = "insert the row if its key is new, otherwise update the existing row". The main tool for idempotent loads.

## A6 · The medallion architecture

We move data through layers. Each layer has **one job** and a strict rule, so you always know where a problem was fixed.

```
 SOURCE SYSTEMS          LANDING           BRONZE               SILVER                  GOLD                  SEMANTIC
 (tills, web, ERP,  -->  data/raw/   -->   raw tables     -->   clean tables     -->   star schema     -->   governed KPIs
  CRM, finance)          files as          everything as        typed, decoded,        facts + dims          (metric views)
                         delivered         text + audit cols    deduplicated,          business rules,       what Genie uses
                         NEVER edited      loaded once          conformed ids,         Unknown members,
                                                                PII separated,         keys + comments
                                                                bad rows quarantined
```

| Layer | Rule of thumb | Who uses it |
|---|---|---|
| **Landing** | Files exactly as delivered. Never edited. | Nobody queries it directly |
| **Bronze** | One table per feed. Everything is text. Add `_source_file`, `_row_number`, `_ingested_at`. Load each file once. | Data engineers, for debugging and replay |
| **Silver** | Clean and conformed. Keep every *valid* source row (even voids, flagged). Reject broken rows with a reason. | Data engineers, data scientists |
| **Gold** | Only what counts for the business, in a star schema with declared grain, keys and comments. | Analysts, dashboards, Genie |
| **Semantic** | Each KPI defined exactly once (net sales, margin %, stock on hand). | Genie, BI tools |

Why so many layers? Because **you will be wrong about something**. If bronze keeps the original data, you can fix a
silver rule and rebuild everything downstream. If you cleaned data on the way in, the original is gone.

## A7 · Star schema: facts and dimensions

Gold is organised as a **star schema**:

* A **fact table** records *events* or *measurements*: things that happened and have numbers you add up
  (a receipt line with its amount, a nightly stock count). Facts are long (many rows) and narrow.
* A **dimension table** describes the *who / what / where / when* of those events (store, product, customer,
  promotion, date). Dimensions are short and wide, with the words people use to filter and group.

```
                 dim_date
                    |
   dim_customer --- fct_sales_line --- dim_store
                    |          \
             dim_promotion   dim_product

   dim_date --- fct_inventory_daily --- dim_store
                        |
                   dim_product
```

Two more ideas you will see:

* **Unknown members**: each dimension gets an explicit row such as `UNKNOWN` (store/product), `ANONYMOUS`
  (customer), `NO_PROMO` (promotion). A fact row whose key can't be matched points there instead of being NULL
  or being dropped. So an inner join never silently loses money.
* **Slowly Changing Dimension type 2 (SCD2)**: when a store's attributes change (Boston was remodelled from a
  Neighborhood Market into a Supercenter), we keep **both versions** with `valid_from` / `valid_to` dates instead
  of overwriting. Built in Silver 03.

## A8 · Measures: additive, semi-additive, non-additive

| Kind | Can you SUM it across... | Example | The trap |
|---|---|---|---|
| **Additive** | anything (stores, products, days) | net sales, units, margin $ | none |
| **Semi-additive** | stores and products, but **not across days** | stock on hand | summing 7 nightly stock counts gives 7x the stock |
| **Non-additive** | nothing: must be *recomputed* from sums | margin %, average basket | averaging line-level percentages gives the wrong answer; use `SUM(margin) / SUM(sales)` |

The semantic layer (Part H) encodes these rules so they are applied the same way every time.

## A9 · Personal data (PII)

Card numbers, emails and dates of birth are **personally identifiable information**. Rules we follow:
replace identifiers with a one-way **hash** (`SHA-256`) so records still join but nobody can read the card number;
replace a date of birth with an **age band**; keep the originals in a separate **restricted** table that never flows
to gold or to the AI.

## A10 · Retail vocabulary used in this notebook

| Term | Meaning |
|---|---|
| **POS / TLOG** | Point of sale: the till. The *transaction log* is its export: one row per line on a receipt |
| **Receipt / basket / transaction** | One customer checkout. Has several **lines** |
| **SKU** | Stock keeping unit: one sellable product (`4471023` = a bag of chips) |
| **Void** | A line scanned and then cancelled at the till. Not a sale |
| **Return** | A product brought back. Negative sales |
| **Training register** | A till (register `99`) used to train staff. Its "sales" are fake |
| **Promotion mechanic** | How a promo works: BOGO (buy one get one), percent off, multi-buy (3 for 2), loyalty price |
| **Gross / discount / net sales** | Shelf-price value / promo discount / what the customer actually paid (excl. tax). Net sales is "sales" |
| **COGS, gross margin** | Cost of goods sold; margin = net sales − COGS |
| **4-5-4 fiscal calendar** | Retail year split into quarters of 4, 5, 4 weeks. Weeks run Sunday–Saturday; the year starts on the Sunday nearest 1 February. Some years have 53 weeks |
| **Like-for-like (LFL) / comparable stores** | Growth measured only on stores that traded a full year in both periods, so new stores don't inflate growth |
| **Dark store** | A warehouse-style store that only fulfils online orders |
| **Shrink / phantom inventory** | Stock lost to theft/damage but not recorded, which can make the system think stock is negative |
| **FX rate** | Foreign exchange rate. Canadian stores sell in CAD; group reporting is in USD |

## A11 · A SQL crash course, on toy data

All the transformations in this notebook are written in **SQL** (Structured Query Language), the language of
databases. The next cells build a tiny in-memory database with 5 sales lines and 2 stores, and demonstrate every SQL
idea you will meet later. Run them and read the output: they are the whole toolkit.

In [1]:
import sqlite3
import pandas as pd

toy = sqlite3.connect(":memory:")
toy.executescript('''
CREATE TABLE sales (receipt TEXT, line INTEGER, store TEXT, item TEXT, qty REAL, amount REAL, margin REAL);
INSERT INTO sales VALUES
  ('R1', 1, '01042', 'A', 2, 6.98, 1.00),
  ('R1', 2, '01042', 'B', 1, 3.99, 2.00),
  ('R2', 1, '01107', 'A', 1, 3.49, 1.40),
  ('R2', 1, '01107', 'A', 1, 3.49, 1.40),   -- the same line delivered twice!
  ('R3', 1, '09999', 'C', 1, 5.00, 0.50);   -- a store that is not in the store list
CREATE TABLE stores (store TEXT PRIMARY KEY, name TEXT, region TEXT);
INSERT INTO stores VALUES ('1042', 'Boston', 'Northeast'), ('1107', 'Hartford', 'Northeast');
''')
tq = lambda sql: pd.read_sql_query(sql, toy)
tq("SELECT * FROM sales")

,receipt,line,store,item,qty,amount,margin
0,R1,1,01042,A,2.0,6.98,1.0
1,R1,2,01042,B,1.0,3.99,2.0
2,R2,1,01107,A,1.0,3.49,1.4
3,R2,1,01107,A,1.0,3.49,1.4
4,R3,1,09999,C,1.0,5.00,0.5


**SELECT / WHERE / expressions.** Pick columns, filter rows, compute new columns. `ltrim(x, '0')` removes leading zeros.

In [2]:
tq("SELECT receipt, store, ltrim(store, '0') AS store_id, amount * 1.10 AS amount_plus_10pct FROM sales WHERE qty >= 1")

,receipt,store,store_id,amount_plus_10pct
0,R1,01042,1042,7.678
1,R1,01042,1042,4.389
2,R2,01107,1107,3.839
3,R2,01107,1107,3.839
4,R3,09999,9999,5.500


**JOIN: why ids must be conformed.** The sales use `01042`, the store list uses `1042`. An **INNER JOIN** keeps only
rows that match on both sides, so this returns **nothing**:

In [3]:
tq("SELECT s.receipt, s.store, st.name FROM sales s JOIN stores st ON st.store = s.store")

,receipt,store,name


Normalise the id first and the join works. But look: receipt `R3` (store `09999`) **disappears**, because an inner
join drops rows with no match. That's $5.00 of sales silently lost.

In [4]:
tq("SELECT s.receipt, s.store, st.name, s.amount FROM sales s JOIN stores st ON st.store = ltrim(s.store, '0')")

,receipt,store,name,amount
0,R1,01042,Boston,6.98
1,R1,01042,Boston,3.99
2,R2,01107,Hartford,3.49
3,R2,01107,Hartford,3.49


A **LEFT JOIN** keeps every row from the left table and fills the missing side with NULL. `COALESCE(a, b)` returns the
first non-NULL value, so we can map unmatched rows to an explicit **Unknown member** instead of losing them.
This is exactly what the gold fact tables do.

In [5]:
tq('''SELECT s.receipt, s.store, COALESCE(st.name, 'Unknown store') AS store_name, s.amount
      FROM sales s LEFT JOIN stores st ON st.store = ltrim(s.store, '0')''')

,receipt,store,store_name,amount
0,R1,01042,Boston,6.98
1,R1,01042,Boston,3.99
2,R2,01107,Hartford,3.49
3,R2,01107,Hartford,3.49
4,R3,09999,Unknown store,5.00


**GROUP BY and aggregates.** `SUM`, `COUNT`, `COUNT(DISTINCT ...)`, `AVG`, `MIN`, `MAX` collapse many rows into one per group.
Notice Hartford's sales are **double counted** (6.98 instead of 3.49) because of the duplicated line.

In [6]:
tq('''SELECT ltrim(store, '0') AS store_id, COUNT(*) AS lines, COUNT(DISTINCT receipt) AS receipts, SUM(amount) AS sales
      FROM sales GROUP BY ltrim(store, '0')''')

,store_id,lines,receipts,sales
0,1042,2,1,10.97
1,1107,2,1,6.98
2,9999,1,1,5.00


**Window functions: ROW_NUMBER() for deduplication.** A *window function* computes a value for each row by looking at
a group ("partition") of related rows, **without collapsing them**. `ROW_NUMBER() OVER (PARTITION BY key ORDER BY ...)`
numbers the copies of each key 1, 2, 3...; keeping `rn = 1` keeps one copy. Used in Silver 01, 04, 06, 07 and 09.

In [7]:
tq('''SELECT receipt, line, store, amount,
             ROW_NUMBER() OVER (PARTITION BY store, receipt, line ORDER BY amount) AS rn
      FROM sales''')

,receipt,line,store,amount,rn
0,R1,1,01042,6.98,1
1,R1,2,01042,3.99,1
2,R2,1,01107,3.49,1
3,R2,1,01107,3.49,2
4,R3,1,09999,5.00,1


**CTEs (`WITH ... AS`)** give a name to an intermediate result so a long query reads top to bottom as a sequence of
steps. Every pipeline file uses them. Here: step 1 numbers the copies, step 2 keeps the first.

In [8]:
tq('''WITH numbered AS (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY store, receipt, line ORDER BY amount) AS rn FROM sales
      ),
      deduplicated AS (
        SELECT * FROM numbered WHERE rn = 1
      )
      SELECT ltrim(store, '0') AS store_id, SUM(amount) AS sales FROM deduplicated GROUP BY 1''')

,store_id,sales
0,1042,10.97
1,1107,3.49
2,9999,5.00


**CASE WHEN** turns codes into words and applies rules row by row.

In [9]:
tq('''SELECT item, CASE item WHEN 'A' THEN 'Chips' WHEN 'B' THEN 'Cookies' ELSE 'Other' END AS item_name,
             CASE WHEN qty > 1 THEN 'multi' ELSE 'single' END AS size
      FROM sales''')

,item,item_name,size
0,A,Chips,multi
1,B,Cookies,single
2,A,Chips,single
3,A,Chips,single
4,C,Other,single


**LAG() and LEAD()** read the previous / next row in a window. They are how we turn a list of dated values into
**validity ranges** (`valid_from` → `valid_to`), used for product costs and for the store history (SCD2).

In [10]:
toy.executescript('''
DROP TABLE IF EXISTS cost;
CREATE TABLE cost (item TEXT, effective_date TEXT, unit_cost REAL);
INSERT INTO cost VALUES ('A', '2025-01-01', 1.00), ('A', '2025-07-01', 1.20), ('A', '2026-03-01', 1.25);
''')
tq('''SELECT item, effective_date AS valid_from,
             COALESCE(date(LEAD(effective_date) OVER (PARTITION BY item ORDER BY effective_date), '-1 day'), '9999-12-31') AS valid_to,
             unit_cost,
             LAG(unit_cost) OVER (PARTITION BY item ORDER BY effective_date) AS previous_cost
      FROM cost''')

,item,valid_from,valid_to,unit_cost,previous_cost
0,A,2025-01-01,2025-06-30,1.00,NaN
1,A,2025-07-01,2026-02-28,1.20,1.0
2,A,2026-03-01,9999-12-31,1.25,1.2


**UPSERT (`INSERT ... ON CONFLICT DO UPDATE`).** Insert a row, or update it if the primary key already exists.
Running it twice gives the same result: that's **idempotency**. On Databricks the same thing is written `MERGE INTO`.

In [11]:
toy.executescript('''
DROP TABLE IF EXISTS dim_store_demo;
CREATE TABLE dim_store_demo (store_id TEXT PRIMARY KEY, name TEXT);
''')
upsert = '''INSERT INTO dim_store_demo VALUES ('1042', 'Boston Seaport'), ('1107', 'Hartford West')
            ON CONFLICT (store_id) DO UPDATE SET name = excluded.name'''
toy.execute(upsert); toy.execute(upsert); toy.execute(upsert)       # three runs...
tq("SELECT * FROM dim_store_demo")                                  # ...still two rows

,store_id,name
0,1042,Boston Seaport
1,1107,Hartford West


**Ratio of sums vs average of ratios.** Margin % must be `SUM(margin) / SUM(amount)`. Averaging each line's
percentage gives equal weight to a $3 line and a $300 line and is simply wrong.

In [12]:
tq('''WITH clean AS (SELECT DISTINCT * FROM sales)
      SELECT ROUND(AVG(margin / amount), 4)       AS wrong_avg_of_line_pcts,
             ROUND(SUM(margin) / SUM(amount), 4)  AS right_ratio_of_sums
      FROM clean''')

,wrong_avg_of_line_pcts,right_ratio_of_sums
0,0.2864,0.2518


That's all the SQL you need. One more piece of theory before we start: **why the steps run in a fixed order**.
Later tables are built from earlier ones (sales need FX rates, FX rates need the calendar, facts need dimensions),
so the pipeline is a **dependency graph** that must run in order:

```
bronze (all 15 feeds)
  └─ silver 01 fiscal_calendar
       ├─ silver 02 fx_rate_daily ─────────────┬─ silver 07 pos_sales_line (+ quarantine)
       │                                       └─ silver 08 online_sales_line
       ├─ silver 03 store_history
       ├─ silver 04 product, product_cost
       ├─ silver 05 promotion, promotion_item
       ├─ silver 06 loyalty_member (+ restricted)
       └─ silver 09 inventory_daily
gold: 01 dim_date → 02 dim_store → 03 dim_product → 04 dim_promotion → 05 dim_customer
      → 06 fct_sales_line → 07 fct_inventory_daily        (dimensions first, facts last)
semantic: sales_metrics, inventory_metrics, fn_like_for_like_sales
```

---
# Part B · Setup: an empty warehouse

The local demo uses **SQLite**, a database engine built into Python, so there is nothing to install.
Each layer is its own database file in `warehouse/` (`bronze.db`, `silver.db`, `gold.db`), attached under the
layer's name. That lets SQL say `bronze.pos_tlog_raw` or `gold.fct_sales_line`, just like Databricks says
`freshcart.bronze.pos_tlog_raw`.

The connection also registers three small functions so the SQL reads like Databricks SQL:
`try_to_date(text, format)` (a **strict** date parser that returns NULL for impossible dates),
`sha2(text, 256)` (a one-way hash) and `initcap(text)` (Title Case).

The next cell **deletes any existing warehouse** and starts from nothing.

In [13]:
import os, sys, json, csv, sqlite3
from pathlib import Path
from datetime import date

# Find the repository root whether the notebook is started from notebooks/ or from the root
ROOT = Path.cwd()
while not (ROOT / "freshcart").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import pandas as pd
from freshcart import config as C, bronze, contracts, quality, metrics
from freshcart.db import connect, render_sql, run_sql_file

pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 250)

con = connect(fresh_layers=C.LAYERS)          # delete warehouse/*.db and attach three empty layers

def q(sql, params=()):
    '''Run a query against the warehouse and show the result as a table.'''
    return pd.read_sql_query(render_sql(sql), con, params=params).convert_dtypes()   # keep integers as integers

def n(sql):
    '''Run a query that returns one number.'''
    return con.execute(render_sql(sql)).fetchone()[0]

def show_sql(path):
    '''Print a pipeline SQL file so you can read it before it runs.'''
    print((ROOT / path).read_text(encoding="utf-8"))

def run_step(path):
    '''Run one pipeline SQL file against the warehouse.'''
    run_sql_file(con, ROOT / path)
    print(f"ran {path}")

def layer_tables(layer):
    names = [r[0] for r in con.execute(f"SELECT name FROM {layer}.sqlite_master WHERE type = 'table' ORDER BY name")]
    return pd.DataFrame([(t, n(f'SELECT COUNT(*) FROM {layer}."{t}"')) for t in names], columns=[f"{layer} table", "rows"])

print("Repository root :", ROOT)
print("SQLite version  :", sqlite3.sqlite_version)
print("As-of date      :", C.AS_OF_DATE, "(the pipeline's 'today'; pinned so every number is reproducible)")
print("Tables per layer:", {layer: len(layer_tables(layer)) for layer in C.LAYERS})

Repository root : /home/user/ai-ready-retail-data-genie-demo
SQLite version  : 3.45.1
As-of date      : 2026-09-27 (the pipeline's 'today'; pinned so every number is reproducible)
Tables per layer: {'bronze': 0, 'silver': 0, 'gold': 0}


Zero tables in every layer. Everything that follows is built by the cells below.

**About the as-of date.** In production the pipeline would use today's date. Here "today" is pinned to
**Sunday 27 September 2026**, so "yesterday" is 26 September, "last week" is fiscal week 34 of FY2026, and every
result in this notebook is the same on every machine.

---
# Part C · Start from the business questions

AI-ready data starts from the **questions people will ask**, not from the tables. They decide what grain the facts
need, which attributes the dimensions need, and which metrics must be defined once. FreshCart is a fictional grocer
with 9 stores in the US and Canada, an online shop, a loyalty programme and weekly promotions.

| # | Question | What the data must provide | Built in |
|---|---|---|---|
| 1 | What were net sales and margin by region last week? | one definition of net sales; cost for margin; a "last completed fiscal week" flag | Silver 04, Gold 01, 06 |
| 2 | How are we trading year to date versus last year? | the same *fiscal* days last year | Silver 01, Gold 01 |
| 3 | What is like-for-like growth this year by region? | a comparable-store rule; a trusted calculation | Gold 02, Semantic |
| 4 | Which categories rely most on promotions this quarter? | a promotion flag per line; a ratio measure | Silver 05, Gold 06 |
| 5 | Rank the Snacks subcategories by margin % | decoded product hierarchy; ratio of sums | Silver 04, Semantic |
| 6 | Do Gold members spend more per basket than Silver? | loyalty tier without personal data; basket count | Silver 06, Gold 05 |
| 7 | What share of sales is online, by region? | in-store and online in one fact with a channel column | Silver 07, 08, Gold 06 |
| 8 | How did Boston Seaport trade week by week this year? | store names people type | Silver 03, Gold 02 |
| 9 | What is the return rate in Household over the last four weeks? | returns as negative lines | Silver 07, Gold 06 |
| 10 | Where are we out of stock most often in fresh departments? | daily stock snapshots; an out-of-stock definition | Silver 09, Gold 07 |
| 11 | How much Produce stock did we hold at the end of last week? | stock treated as semi-additive | Gold 07, Semantic |
| 12 | Which stores opened in FY2025? | opening dates in fiscal years | Gold 02 |

Three real-world events happen in the data and must be handled correctly:
1. **Boston Seaport (1042) was remodelled** from a Neighborhood Market into a Supercenter on 1 Feb 2026.
2. **Mississauga Square One (322) opened** on 14 June 2025, so it's not comparable until FY2027.
3. **Atlanta Buckhead (2211) closed** on 30 May 2026.

---
# Part D · The landing zone: raw files as delivered

The **landing zone** (`data/raw/`) holds files exactly as the source systems deliver them. We **never edit** them.
Here is every file, grouped by source system:

In [14]:
files = sorted(p for p in C.RAW_DIR.rglob("*") if p.is_file() and p.name != "README.md")
inventory = pd.DataFrame([(p.parent.name, p.name, p.suffix, round(p.stat().st_size / 1024, 1)) for p in files],
                         columns=["source system", "file", "format", "KB"])
print(f"{len(files)} files in data/raw/")
inventory.groupby(["source system", "format"]).agg(files=("file", "count"), total_KB=("KB", "sum")).reset_index()

59 files in data/raw/


,source system,format,files,total_KB
0,crm,.csv,1,88.2
1,ecommerce,.ndjson,20,7081.8
2,erp,.csv,3,12.9
3,finance,.csv,2,300.7
4,inventory,.csv,4,1767.5
5,pos,.csv,20,8538.2
6,promotions,.csv,2,17.4
7,stores,.csv,7,3.0


Let's open a few files as plain text, the way a data engineer does before writing any code. Look for the problems
before reading the notes under each one.

In [15]:
def peek(rel_path, n_lines=4):
    print(f"--- data/raw/{rel_path}")
    with open(C.RAW_DIR / rel_path, encoding="utf-8") as fh:
        for _ in range(n_lines):
            print(fh.readline().rstrip("\n")[:220])
    print()

peek("pos/pos_tlog_202609.csv")
peek("stores/store_master_20260601.csv")
peek("finance/fiscal_calendar.csv")
peek("erp/product_master.csv")
peek("ecommerce/ecom_orders_202609.ndjson", 1)

--- data/raw/pos/pos_tlog_202609.csv
TRX_ID,STR_NBR,REG_NBR,LN_NBR,TRX_DT,TRX_TM,ITM_ID,QTY,UNIT_PRC,EXT_AMT,DISC_AMT,TAX_AMT,TRX_TYP,LYL_CARD_NBR,PROMO_CD,CRNCY_CD,TNDR_CD
70032179,01042,04,1,20260901,123616,000004493002,2,3.99,7.98,3.99,0.25,S,,BG26W31,USD,CS
70032179,01042,04,2,20260901,123616,000004481003,1,4.99,4.99,0.00,0.31,S,,,USD,CS
70032179,01042,04,3,20260901,123616,000000030201,1,9.99,9.99,0.00,0.00,S,,,USD,CS

--- data/raw/stores/store_master_20260601.csv
STR_NBR,STR_NM,CITY,ST_CD,CNTRY,FMT_CD,RGN_CD,OPEN_DT,CLOSE_DT,SQFT
01042,FreshCart Boston Seaport,Boston,MA,US,SC,02,05/14/2016,,52000
01107,FreshCart Hartford West,West Hartford,CT,US,EXP,02,10/05/2019,,9500
09001,FreshCart Newark Fulfilment Center,Newark,NJ,US,DRK,02,03/11/2023,,0

--- data/raw/finance/fiscal_calendar.csv
CAL_DT,FISC_YR,FISC_QTR,FISC_PRD,FISC_PRD_NM,FISC_WK,FISC_WK_START
31/01/2010,FY2010,Q1,P01,Feb,W01,31/01/2010
01/02/2010,FY2010,Q1,P01,Feb,W01,31/01/2010
02/02/2010,FY2010,Q1,P01,Feb,W01,31/01/2010



Already visible:

* **POS**: store `01042` and item `000004493002` are zero-padded; the date is `20260901` and time `123616` (strings);
  there are three amount columns (`EXT_AMT`, `DISC_AMT`, `TAX_AMT`), so which one is "sales"?
* **Store master**: dates are US-style `05/14/2016` (month first), format and region are codes (`SC`, `02`).
* **Fiscal calendar**: dates are `31/01/2010` (day first, the opposite of the store master!) and years are labels like `FY2010`.
* **Product master**: the Canadian copy (`CA01`) is in UPPER CASE; the category is a code (`P0203`).
* **Online orders**: JSON with the order **lines nested** inside, and a timestamp with a timezone offset (`-04:00`).

Each problem is fixed, deliberately and visibly, in silver.

---
# Part E · Bronze: land every file exactly as delivered

### The rules of bronze
1. **One table per feed.** All `pos_tlog_*.csv` files go into `bronze.pos_tlog_raw`, and so on.
2. **Every column is stored as text.** Nothing is cast, renamed, trimmed or cleaned. `01042` stays `01042`,
   `20250230` stays `20250230`. Automatic type guessing would quietly "fix" things (drop leading zeros, guess
   dates); fixing is silver's job, done deliberately.
3. **Three audit columns** on every row: `_source_file` (which file), `_row_number` (which row in that file),
   `_ingested_at` (when it landed). Any number in gold can be traced back to an exact file and line.
4. **Each file is loaded once.** Every loaded file is recorded in `bronze._ingestion_log`; files already there are
   skipped. That makes ingestion **incremental and idempotent**. (On Databricks, Auto Loader / `read_files` with a
   checkpoint gives the same guarantee.)
5. **JSON is kept whole.** Each online order is stored as one text value; silver parses it.

**Why not clean the data on the way in?** Because if bronze holds the original bytes, you can fix a silver rule and
rebuild everything. If bronze was "cleaned", the original is gone. Bronze is cheap insurance.

The loader is `freshcart/bronze.py`. Its list of feeds maps each file pattern to a bronze table:

In [16]:
pd.DataFrame(bronze.feed_catalog(), columns=["bronze table", "file pattern in data/raw", "format", "what it is"])

,bronze table,file pattern in data/raw,format,what it is
0,pos_tlog_raw,pos/pos_tlog_*.csv,csv,"POS transaction log, one file per month"
1,ecom_order_raw,ecommerce/ecom_orders_*.ndjson,ndjson,"Online orders, one JSON document per line"
2,product_master_raw,erp/product_master.csv,csv,"ERP material master, one row per SKU per sales org"
3,merch_hierarchy_raw,erp/merch_hierarchy.csv,csv,Material group code to department/category/subcategory
4,cost_history_raw,erp/cost_history.csv,csv,Standard cost with effective dates
5,store_master_raw,stores/store_master_*.csv,csv,Monthly snapshots of the store master
6,ref_region_raw,stores/ref_region.csv,csv,Region code lookup
7,ref_store_format_raw,stores/ref_store_format.csv,csv,Store format code lookup
8,ref_state_province_raw,stores/ref_state_province.csv,csv,State/province code lookup
9,crm_member_raw,crm/crm_members_*.csv,csv,Loyalty CRM member extract (contains personal data)


### Run the bronze load

In [17]:
bronze_stats = bronze.ingest(con)

  bronze.pos_tlog_raw              20 new of  20 files    98,192 rows
  bronze.ecom_order_raw            20 new of  20 files     8,288 rows
  bronze.product_master_raw         1 new of   1 files       162 rows
  bronze.merch_hierarchy_raw        1 new of   1 files        30 rows
  bronze.cost_history_raw           1 new of   1 files       114 rows
  bronze.store_master_raw           4 new of   4 files        35 rows
  bronze.ref_region_raw             1 new of   1 files         5 rows
  bronze.ref_store_format_raw       1 new of   1 files         4 rows
  bronze.ref_state_province_raw     1 new of   1 files         7 rows
  bronze.crm_member_raw             1 new of   1 files     1,367 rows
  bronze.promo_calendar_raw         1 new of   1 files       127 rows
  bronze.promo_item_raw             1 new of   1 files       575 rows
  bronze.fiscal_calendar_raw        1 new of   1 files     6,573 rows
  bronze.fx_rate_raw                1 new of   1 files       446 rows


  bronze.inventory_raw              4 new of   4 files    52,052 rows


In [18]:
layer_tables("bronze")

,bronze table,rows
0,_ingestion_log,59
1,cost_history_raw,114
2,crm_member_raw,1367
3,ecom_order_raw,8288
4,fiscal_calendar_raw,6573
5,fx_rate_raw,446
6,inventory_raw,52052
7,merch_hierarchy_raw,30
8,pos_tlog_raw,98192
9,product_master_raw,162


### The ingestion log (how bronze knows what it has already loaded)

In [19]:
q("SELECT source_file, target_table, row_count FROM bronze._ingestion_log ORDER BY target_table, source_file").head(12)

,source_file,target_table,row_count
0,erp/cost_history.csv,cost_history_raw,114
1,crm/crm_members_20260906.csv,crm_member_raw,1367
2,ecommerce/ecom_orders_202502.ndjson,ecom_order_raw,363
3,ecommerce/ecom_orders_202503.ndjson,ecom_order_raw,410
4,ecommerce/ecom_orders_202504.ndjson,ecom_order_raw,433
5,ecommerce/ecom_orders_202505.ndjson,ecom_order_raw,420
6,ecommerce/ecom_orders_202506.ndjson,ecom_order_raw,395
7,ecommerce/ecom_orders_202507.ndjson,ecom_order_raw,446
8,ecommerce/ecom_orders_202508.ndjson,ecom_order_raw,440
9,ecommerce/ecom_orders_202509.ndjson,ecom_order_raw,404


### Proof 1: everything in bronze is text

SQLite's `typeof()` tells us the stored type. Every source column is `text`, including quantities, amounts and dates.
Only our own audit column `_row_number` is an integer.

In [20]:
q("SELECT typeof(TRX_DT) AS TRX_DT, typeof(QTY) AS QTY, typeof(EXT_AMT) AS EXT_AMT, typeof(STR_NBR) AS STR_NBR, typeof(_row_number) AS _row_number FROM bronze.pos_tlog_raw LIMIT 1")

,TRX_DT,QTY,EXT_AMT,STR_NBR,_row_number
0,text,text,text,text,integer


### Proof 2: bronze is incremental and idempotent

Run the loader again. No file is new, so **nothing** is loaded and no row is duplicated. When next month's POS file
lands in `data/raw/pos/`, only that file will be loaded.

In [21]:
before = n("SELECT COUNT(*) FROM bronze.pos_tlog_raw")
bronze.ingest(con, verbose=False)
after = n("SELECT COUNT(*) FROM bronze.pos_tlog_raw")
print(f"POS rows before the re-run: {before:,}   after: {after:,}   -> duplicated: {after - before}")

POS rows before the re-run: 98,192   after: 98,192   -> duplicated: 0


## E.1 · Every bronze table, one by one

For each table: what the source is, what one row means, what's wrong with it, and which silver step fixes it.
We only **look** here; bronze never changes anything.

### `bronze.pos_tlog_raw` · the till transaction log
**Source:** 20 monthly CSV files from the store POS systems. **Grain:** one row per line on a receipt, as the till wrote it.
Here is the "golden receipt" we'll follow through every layer (store `01042`, receipt `88120431`) and a Canadian return (`88120519`):

In [22]:
q('''SELECT TRX_ID, STR_NBR, REG_NBR, LN_NBR, TRX_DT, TRX_TM, ITM_ID, QTY, UNIT_PRC, EXT_AMT, DISC_AMT, TAX_AMT,
            TRX_TYP, LYL_CARD_NBR, PROMO_CD, CRNCY_CD, _source_file, _row_number
     FROM bronze.pos_tlog_raw WHERE TRX_ID IN ('88120431', '88120519') ORDER BY TRX_ID, CAST(LN_NBR AS INTEGER)''')

,TRX_ID,STR_NBR,REG_NBR,LN_NBR,TRX_DT,TRX_TM,ITM_ID,QTY,UNIT_PRC,EXT_AMT,DISC_AMT,TAX_AMT,TRX_TYP,LYL_CARD_NBR,PROMO_CD,CRNCY_CD,_source_file,_row_number
0,88120431,01042,03,1,20260914,93015,000004471023,2,3.49,6.98,3.49,0.22,S,6034118822,BG26W33,USD,pos/pos_tlog_202609.csv,2161
1,88120431,01042,03,2,20260914,93015,000000093321,1.254,1.49,1.87,0.00,0.00,S,6034118822,,USD,pos/pos_tlog_202609.csv,2162
2,88120431,01042,03,3,20260914,93015,000004471023,1,3.49,3.49,0.00,0.00,V,6034118822,,USD,pos/pos_tlog_202609.csv,2163
3,88120431,01042,03,4,20260914,93015,000000005012,1,3.99,3.99,0.00,0.00,S,6034118822,,USD,pos/pos_tlog_202609.csv,2164
4,88120519,00317,01,1,20260914,94210,000000551020,1,18.99,18.99,0.00,0.00,R,,,CAD,pos/pos_tlog_202609.csv,2281


Problems planted in this feed (count them yourself below): zero-padded ids; dates and times as strings (`93015` means
09:30:15); `TRX_TYP` is a code (S = Sale, R = Return, **V = Void**); **returns arrive with positive amounts**;
register `99` is the **training till**; one day was **delivered twice**; some rows have **no store**, an **impossible
date** (30 February) or a **scanner error** quantity of 9999; one item isn't in the product master; the loyalty card
number is **personal data**; Canadian stores sell in **CAD**. → Fixed in **Silver 07**.

In [23]:
q('''SELECT 'Voided lines (TRX_TYP = V)' AS problem, COUNT(*) AS rows FROM bronze.pos_tlog_raw WHERE TRX_TYP = 'V'
     UNION ALL SELECT 'Return lines (arrive with positive amounts)', COUNT(*) FROM bronze.pos_tlog_raw WHERE TRX_TYP = 'R'
     UNION ALL SELECT 'Training register 99', COUNT(*) FROM bronze.pos_tlog_raw WHERE REG_NBR = '99'
     UNION ALL SELECT 'Missing store number', COUNT(*) FROM bronze.pos_tlog_raw WHERE TRIM(STR_NBR) = ''
     UNION ALL SELECT 'Impossible date (30 February)', COUNT(*) FROM bronze.pos_tlog_raw WHERE substr(TRX_DT, 5) = '0230'
     UNION ALL SELECT 'Quantity 9999 (scanner error)', COUNT(*) FROM bronze.pos_tlog_raw WHERE QTY = '9999'
     UNION ALL SELECT 'Lines in Canadian dollars', COUNT(*) FROM bronze.pos_tlog_raw WHERE CRNCY_CD = 'CAD'
     UNION ALL SELECT 'Lines with a loyalty card (personal data)', COUNT(*) FROM bronze.pos_tlog_raw WHERE LYL_CARD_NBR <> ''
''')

,problem,rows
0,Voided lines (TRX_TYP = V),357
1,Return lines (arrive with positive amounts),428
2,Training register 99,238
3,Missing store number,18
4,Impossible date (30 February),12
5,Quantity 9999 (scanner error),9
6,Lines in Canadian dollars,24232
7,Lines with a loyalty card (personal data),40600


### `bronze.ecom_order_raw` · online orders
**Source:** 20 monthly NDJSON files (one JSON document per line). **Grain:** one row per **order**, with the lines still
nested inside the `value` column. Problems: lines must be **flattened**; the store id has **no** leading zeros
(`9001`), unlike POS; the timestamp has a **UTC offset**; cancelled orders must not count. → Fixed in **Silver 08**.

In [24]:
first_order = n("SELECT value FROM bronze.ecom_order_raw ORDER BY _source_file DESC, _row_number LIMIT 1")
print(json.dumps(json.loads(first_order), indent=2)[:900], "...")
q("SELECT json_extract(value, '$.status') AS status, COUNT(*) AS orders FROM bronze.ecom_order_raw GROUP BY 1")

{
  "orderId": "W1007923",
  "orderTs": "2026-09-01T23:25:38-04:00",
  "fulfilmentStoreId": "9001",
  "loyaltyId": 6097499564,
  "currency": "USD",
  "status": "DELIVERED",
  "lines": [
    {
      "lineNo": 1,
      "sku": "20103",
      "qty": 1,
      "unitPrice": 4.99,
      "lineTotal": 4.99,
      "discount": 1.5,
      "promoCode": "PC26W31"
    },
    {
      "lineNo": 2,
      "sku": "4493002",
      "qty": 1,
      "unitPrice": 3.99,
      "lineTotal": 3.99,
      "discount": 0.0,
      "promoCode": null
    },
    {
      "lineNo": 3,
      "sku": "5013",
      "qty": 1,
      "unitPrice": 3.89,
      "lineTotal": 3.89,
      "discount": 0.0,
      "promoCode": null
    },
    {
      "lineNo": 4,
      "sku": "552001",
      "qty": 1,
      "unitPrice": 4.49,
      "lineTotal": 4.49,
      "discount": 0.0,
      "promoCode": null
    },
    {
      "lineNo": 5,
      "sku": " ...


,status,orders
0,CANCELLED,248
1,DELIVERED,8040


### `bronze.product_master_raw` · ERP product (material) master
**Grain:** one row per SKU **per sales organisation** (`US01` and `CA01`). So most products appear **twice**. Problems:
duplicate rows per product (would double sales in a join); the category `MATKL` is a code; the Canadian copy is upper
case and never marks items discontinued; two items exist only in Canada. → Fixed in **Silver 04**.

In [25]:
print("rows:", n("SELECT COUNT(*) FROM bronze.product_master_raw"),
      "| distinct SKUs:", n("SELECT COUNT(DISTINCT ltrim(MATNR, '0')) FROM bronze.product_master_raw"))
q("SELECT MATNR, VKORG, MAKTX, MATKL, BRAND_NM, MEINS, STATUS FROM bronze.product_master_raw WHERE MATNR IN ('4471023', '4471024', '4492050') ORDER BY MATNR, VKORG")

rows: 162 | distinct SKUs: 82


,MATNR,VKORG,MAKTX,MATKL,BRAND_NM,MEINS,STATUS
0,4471023,CA01,CRUNCHY SEA SALT CHIPS 200G,C0412,Crunchwell,EA,A
1,4471023,US01,Crunchy Sea Salt Chips 200g,C0412,Crunchwell,EA,A
2,4471024,CA01,BBQ KETTLE CHIPS 200G,C0412,Crunchwell,EA,A
3,4471024,US01,BBQ Kettle Chips 200g,C0412,Crunchwell,EA,D
4,4492050,CA01,MAPLE BAKED BEANS 398ML,C0432,Pantry Lane,EA,A


### `bronze.merch_hierarchy_raw` · category code lookup
**Grain:** one row per material group code. Translates `C0412` into Department / Category / Subcategory. Used by **Silver 04**.

In [26]:
q("SELECT MATKL, DEPT_NM, CAT_NM, SUBCAT_NM FROM bronze.merch_hierarchy_raw ORDER BY MATKL").head(8)

,MATKL,DEPT_NM,CAT_NM,SUBCAT_NM
0,B0101,Bakery,Bread,Sliced Bread
1,B0201,Bakery,Sweet Bakery,Pastries and Muffins
2,C0412,Grocery,Snacks,Chips and Crisps
3,C0413,Grocery,Snacks,Cookies and Biscuits
4,C0414,Grocery,Snacks,Chocolate and Candy
5,C0421,Grocery,Beverages,Soft Drinks
6,C0422,Grocery,Beverages,Coffee and Tea
7,C0423,Grocery,Beverages,Bottled Water


### `bronze.cost_history_raw` · standard cost with effective dates
**Grain:** one row per SKU per cost change. Only a start date (`COST_EFF_DT`, `yyyyMMdd`) is given; the end date is
implied by the next change. → Turned into validity ranges in **Silver 04** (`product_cost`).

In [27]:
q("SELECT MATNR, COST_EFF_DT, STD_COST, CRNCY_CD FROM bronze.cost_history_raw WHERE MATNR IN ('20101', '20201') ORDER BY MATNR, COST_EFF_DT")

,MATNR,COST_EFF_DT,STD_COST,CRNCY_CD
0,20101,20240101,1.10,USD
1,20101,20251228,1.14,USD
2,20201,20240101,2.42,USD
3,20201,20260713,2.53,USD


### `bronze.store_master_raw` + three code lookups
**Source:** 4 snapshot files of the store master, one delivered each time something changed (the snapshot date is only
in the **file name**). **Grain:** one row per store per snapshot. Problems: codes for format, region and state; US-style
`MM/DD/YYYY` dates; leading zeros; history must be **derived** by comparing snapshots. → Fixed in **Silver 03**.

The lookups `ref_store_format_raw`, `ref_region_raw`, `ref_state_province_raw` translate those codes.

In [28]:
q('''SELECT _source_file, STR_NBR, STR_NM, FMT_CD, RGN_CD, ST_CD, CNTRY, OPEN_DT, CLOSE_DT, SQFT
     FROM bronze.store_master_raw WHERE STR_NBR IN ('01042', '02211') ORDER BY STR_NBR, _source_file''')

,_source_file,STR_NBR,STR_NM,FMT_CD,RGN_CD,ST_CD,CNTRY,OPEN_DT,CLOSE_DT,SQFT
0,stores/store_master_20250201.csv,01042,FreshCart Boston Seaport,NM,02,MA,US,05/14/2016,,38000
1,stores/store_master_20250601.csv,01042,FreshCart Boston Seaport,NM,02,MA,US,05/14/2016,,38000
2,stores/store_master_20260201.csv,01042,FreshCart Boston Seaport,SC,02,MA,US,05/14/2016,,52000
3,stores/store_master_20260601.csv,01042,FreshCart Boston Seaport,SC,02,MA,US,05/14/2016,,52000
4,stores/store_master_20250201.csv,02211,FreshCart Atlanta Buckhead,EXP,01,GA,US,03/18/2017,,8800
5,stores/store_master_20250601.csv,02211,FreshCart Atlanta Buckhead,EXP,01,GA,US,03/18/2017,,8800
6,stores/store_master_20260201.csv,02211,FreshCart Atlanta Buckhead,EXP,01,GA,US,03/18/2017,,8800
7,stores/store_master_20260601.csv,02211,FreshCart Atlanta Buckhead,EXP,01,GA,US,03/18/2017,05/30/2026,8800


In [29]:
display(q("SELECT FMT_CD, FMT_NM FROM bronze.ref_store_format_raw"))
display(q("SELECT RGN_CD, RGN_NM FROM bronze.ref_region_raw"))
display(q("SELECT ST_CD, ST_NM, CNTRY FROM bronze.ref_state_province_raw"))

,FMT_CD,FMT_NM
0,DRK,Dark Store
1,EXP,Express
2,NM,Neighborhood Market
3,SC,Supercenter


,RGN_CD,RGN_NM
0,01,Southeast
1,02,Northeast
2,03,Midwest
3,04,West
4,05,Canada


,ST_CD,ST_NM,CNTRY
0,CO,Colorado,US
1,CT,Connecticut,US
2,GA,Georgia,US
3,IL,Illinois,US
4,MA,Massachusetts,US
5,NJ,New Jersey,US
6,ON,Ontario,CA


### `bronze.crm_member_raw` · loyalty members (personal data!)
**Source:** a CRM extract dated 6 September 2026 (date in the file name). **Grain:** one row per loyalty card.
Contains card number, email and date of birth, all **personal data** that must never reach gold or the AI. Members who
joined after the extract date are already shopping but not in the file yet (**late-arriving members**).
→ Fixed in **Silver 06** and **Gold 05**.

In [30]:
q("SELECT CARD_NBR, EMAIL, DOB, TIER, ENROLL_DT, HOME_STR, _source_file FROM bronze.crm_member_raw LIMIT 3")

,CARD_NBR,EMAIL,DOB,TIER,ENROLL_DT,HOME_STR,_source_file
0,6034118822,member00001@example.com,1972-11-15,G,2019-04-02,01042,crm/crm_members_20260906.csv
1,6046659051,member00027@example.com,1956-06-23,B,2022-05-24,00317,crm/crm_members_20260906.csv
2,6044925192,member00028@example.com,1975-10-31,B,2022-11-05,00317,crm/crm_members_20260906.csv


### `bronze.promo_calendar_raw` and `bronze.promo_item_raw` · promotions
**Grain:** one row per promotion; one row per promotion per product. Problems: mechanic codes (`BOGO`, `PCT`, `MULTI`,
`LOY`), text dates, an empty string instead of NULL for "no discount %", zero-padded item ids. → **Silver 05**.

In [31]:
display(q("SELECT MECH_CD, COUNT(*) AS promotions, MIN(PROMO_CD) AS example, MIN(PROMO_DESC) AS example_desc FROM bronze.promo_calendar_raw GROUP BY 1"))
display(q("SELECT * FROM bronze.promo_item_raw LIMIT 3"))

,MECH_CD,promotions,example,example_desc
0,BOGO,27,BG25W01,Back to School - Buy one get one free
1,LOY,24,LY25W05,Back to School - Members save 15%
2,MULTI,29,MB25W03,Back to School - 3 for the price of 2
3,PCT,47,PC25W08,Back to School - 20% off


,PROMO_CD,MATNR,_source_file,_row_number,_ingested_at
0,BG25W01,20101,promotions/promo_items.csv,1,2026-09-30T10:36:52+00:00
1,BG25W01,94202,promotions/promo_items.csv,2,2026-09-30T10:36:52+00:00
2,BG25W01,551020,promotions/promo_items.csv,3,2026-09-30T10:36:52+00:00


### `bronze.fiscal_calendar_raw` · Finance's 4-5-4 calendar
**Grain:** one row per calendar day (2010–2027). Problems: `DD/MM/YYYY` dates; text labels (`FY2026`, `Q1`, `P01`, `W34`).
→ **Silver 01**.

In [32]:
q("SELECT CAL_DT, FISC_YR, FISC_QTR, FISC_PRD, FISC_PRD_NM, FISC_WK, FISC_WK_START FROM bronze.fiscal_calendar_raw WHERE CAL_DT IN ('31/01/2026', '01/02/2026', '02/02/2026', '08/02/2026')")

,CAL_DT,FISC_YR,FISC_QTR,FISC_PRD,FISC_PRD_NM,FISC_WK,FISC_WK_START
0,31/01/2026,FY2025,Q4,P12,Jan,W52,25/01/2026
1,01/02/2026,FY2026,Q1,P01,Feb,W01,01/02/2026
2,02/02/2026,FY2026,Q1,P01,Feb,W01,01/02/2026
3,08/02/2026,FY2026,Q1,P01,Feb,W02,08/02/2026


### `bronze.fx_rate_raw` · exchange rates
**Grain:** one row per **business day** per currency pair (CAD→USD). Problem: nothing on weekends and holidays, but a
Saturday sale in Toronto still needs a rate. → **Silver 02**.

In [33]:
q("SELECT RATE_DT, FROM_CCY, TO_CCY, RATE FROM bronze.fx_rate_raw WHERE RATE_DT BETWEEN '2025-09-11' AND '2025-09-16'")

,RATE_DT,FROM_CCY,TO_CCY,RATE
0,2025-09-11,CAD,USD,0.7472
1,2025-09-12,CAD,USD,0.7508
2,2025-09-15,CAD,USD,0.7447
3,2025-09-16,CAD,USD,0.7475


### `bronze.inventory_raw` · nightly stock snapshots
**Source:** 4 monthly files covering the last 13 fiscal weeks. **Grain:** one row per store, item and night. Problems:
text dates and quantities; zero-padded ids; a few **negative** stock readings (physically impossible). → **Silver 09**.

In [34]:
display(q("SELECT STR_NBR, ITM_ID, SNAP_DT, OH_QTY, ON_ORD_QTY FROM bronze.inventory_raw LIMIT 3"))
print("negative stock readings:", n("SELECT COUNT(*) FROM bronze.inventory_raw WHERE CAST(OH_QTY AS REAL) < 0"))

,STR_NBR,ITM_ID,SNAP_DT,OH_QTY,ON_ORD_QTY
0,01042,000000093321,20260628,15.066,0
1,01042,000000093321,20260629,11.691,0
2,01042,000000093321,20260630,10.511,0


negative stock readings: 86


**Bronze summary.** 15 raw tables plus the ingestion log, all text, all traceable, all loaded exactly once, and every
problem still there. That's correct. Now we fix them.

---
# Part F · Silver: clean, type, decode, deduplicate, conform

Silver is where most of the AI-readiness work happens. The rules:

| Rule | Meaning | Example |
|---|---|---|
| **Type everything** | dates are dates, numbers are numbers; impossible values become NULL, never silently "fixed" | `20250230` → NULL → quarantined |
| **One id per entity** (*conforming*) | the same store/product has the same id in every table | `01042`, `1042` → `1042` |
| **Decode codes** | store the words people use | `NM` → `Neighborhood Market` |
| **Deduplicate** | one row per natural key | the re-sent POS day |
| **Keep every valid source row** | voids, cancellations and training sales stay, **flagged**; gold decides what counts | `line_type = 'Void'` |
| **Quarantine broken rows** | rows breaking a hard rule go to a quarantine table **with a reason** | `'missing store number'` |
| **Separate personal data** | pseudonymise; keep originals in a restricted table | `sha2(card)` |

Each silver table is built by one SQL file in `pipeline/silver/`. Every file **drops and rebuilds** its table from
bronze, so silver is always reproducible (the incremental work happens in bronze and gold). For each step below you
will see: the **before**, one transformation at a time, the **real SQL file**, the **after**, and a **check**.

## Silver 01 · `fiscal_calendar`

**From** `bronze.fiscal_calendar_raw` · **Grain** one row per calendar day · **Why first?** Every time-based question
("last week", "P03", "YTD vs last year") and the FX table depend on it.

**Theory: the 4-5-4 retail calendar.** Retailers don't report in calendar months. Months have different numbers of
days and weekdays, so "September vs last September" would compare 4 Saturdays with 5. A 4-5-4 calendar splits the year
into 12 *periods* of 4, 5 and 4 weeks. Every week runs Sunday–Saturday, and the fiscal year starts on the Sunday nearest
1 February. Every period therefore has whole weeks, and "day 50 of this year" is the same weekday as "day 50 of last
year". A year is 364 days, so every 5–6 years a **53rd week** is added to catch up:

In [35]:
q("SELECT FISC_YR, COUNT(*) AS days FROM bronze.fiscal_calendar_raw GROUP BY 1 HAVING COUNT(*) <> 364")

,FISC_YR,days
0,FY2012,371
1,FY2017,371
2,FY2023,371


**Transformation 1: parse the day-first dates strictly.** `try_to_date(text, 'dd/MM/yyyy')` turns `31/01/2026` into the
ISO date `2026-01-31`. ISO dates (`YYYY-MM-DD`) sort correctly as text and are what every engine understands.

**Transformation 2: turn labels into numbers.** `substr('FY2026', 3)` = `2026`, `substr('P01', 2)` = `1`. Numbers can be
compared (`fiscal_year = 2026 - 1`); labels can't.

In [36]:
q('''SELECT CAL_DT, try_to_date(CAL_DT, 'dd/MM/yyyy') AS calendar_date,
            FISC_YR, CAST(substr(FISC_YR, 3) AS INTEGER) AS fiscal_year,
            FISC_PRD, CAST(substr(FISC_PRD, 2) AS INTEGER) AS fiscal_period,
            FISC_WK, CAST(substr(FISC_WK, 2) AS INTEGER) AS fiscal_week
     FROM bronze.fiscal_calendar_raw WHERE CAL_DT IN ('31/01/2026', '01/02/2026')''')

,CAL_DT,calendar_date,FISC_YR,fiscal_year,FISC_PRD,fiscal_period,FISC_WK,fiscal_week
0,31/01/2026,2026-01-31,FY2025,2025,P12,12,W52,52
1,01/02/2026,2026-02-01,FY2026,2026,P01,1,W01,1


**Transformation 3: deduplicate** by keeping the latest delivery of each day (`ROW_NUMBER ... ORDER BY _ingested_at DESC`),
so a re-sent calendar file could never create duplicate days.

**Transformation 4: derive new columns.** `fiscal_day_of_year` (1 on the first Sunday of the fiscal year) makes "same
period last year" a simple comparison of numbers. `day_of_week_name` saves everyone from date arithmetic.

The real SQL file:

In [37]:
show_sql("pipeline/silver/01_fiscal_calendar.sql")

-- =====================================================================================
-- SILVER 01 · fiscal_calendar
-- Source : bronze.fiscal_calendar_raw   (Finance's file, text labels, DD/MM/YYYY dates)
-- Grain  : one row per calendar day
-- Steps  : 1. keep the latest delivery of each day (a re-sent file must not duplicate days)
--          2. parse DD/MM/YYYY strictly
--          3. turn text labels into numbers: 'FY2026' -> 2026, 'P01' -> 1, 'W34' -> 34
--          4. derive fiscal_day_of_year and the weekday name
-- =====================================================================================
DROP TABLE IF EXISTS silver.fiscal_calendar;
CREATE TABLE silver.fiscal_calendar (
  calendar_date          TEXT    PRIMARY KEY,
  fiscal_year            INTEGER NOT NULL,
  fiscal_quarter         INTEGER NOT NULL,
  fiscal_period          INTEGER NOT NULL,
  fiscal_period_name     TEXT    NOT NULL,
  fiscal_week            INTEGER NOT NULL,
  fiscal_week_start_date TEXT    NOT 

In [38]:
run_step("pipeline/silver/01_fiscal_calendar.sql")
q("SELECT * FROM silver.fiscal_calendar WHERE calendar_date IN ('2026-01-31', '2026-02-01', '2026-02-02', '2026-02-08')")

ran pipeline/silver/01_fiscal_calendar.sql


,calendar_date,fiscal_year,fiscal_quarter,fiscal_period,fiscal_period_name,fiscal_week,fiscal_week_start_date,fiscal_day_of_year,day_of_week_name
0,2026-01-31,2025,4,12,P12 Jan,52,2026-01-25,364,Saturday
1,2026-02-01,2026,1,1,P01 Feb,1,2026-02-01,1,Sunday
2,2026-02-02,2026,1,1,P01 Feb,1,2026-02-01,2,Monday
3,2026-02-08,2026,1,1,P01 Feb,2,2026-02-08,8,Sunday


Notice **31 January 2026 is day 364 of FY2025**: the new fiscal year starts on Sunday 1 February 2026.

**Check:** one row per day, no gaps, and every fiscal year starts on a Sunday.

In [39]:
q('''SELECT COUNT(*) AS days, COUNT(DISTINCT calendar_date) AS distinct_days,
            CAST(julianday(MAX(calendar_date)) - julianday(MIN(calendar_date)) AS INTEGER) + 1 AS days_in_range,
            SUM(fiscal_day_of_year = 1 AND day_of_week_name <> 'Sunday') AS years_not_starting_on_sunday
     FROM silver.fiscal_calendar''')

,days,distinct_days,days_in_range,years_not_starting_on_sunday
0,6573,6573,6573,0


## Silver 02 · `fx_rate_daily`

**From** `bronze.fx_rate_raw` + `silver.fiscal_calendar` · **Grain** one row per calendar day per currency ·
**Why:** Canadian sales are in CAD, group reporting is in USD, and a sale on any day needs that day's rate.

**Before:** rates exist only on business days. Nothing for Saturday 13 or Sunday 14 September 2025 (see the bronze
table above).

**Transformation 1: type the published rates** (text → date, text → REAL, upper-case currency codes).

**Transformation 2: build a row for every calendar day** by joining to the calendar, and **carry the last published
rate forward** over weekends and holidays (the correlated subquery `... WHERE rate_date <= calendar_date ORDER BY
rate_date DESC LIMIT 1` means "the most recent rate on or before this day"). Each filled day gets
`is_carried_forward = 1`, so nobody mistakes it for a published rate. This is a standard technique called
**forward fill** or **"last observation carried forward"**.

**Transformation 3: add `USD → USD = 1.0`** for every day. Now *every* sale, in either currency, uses the same join
`ON rate_date = sales_date AND from_currency = currency_code`, with no special case for US stores.

In [40]:
show_sql("pipeline/silver/02_fx_rate_daily.sql")

-- =====================================================================================
-- SILVER 02 · fx_rate_daily
-- Source : bronze.fx_rate_raw   (CAD->USD, published on business days only)
-- Grain  : one row per calendar day per currency
-- Steps  : 1. type the published rates
--          2. build a row for EVERY calendar day, carrying the last published rate forward
--             over weekends and holidays (a Saturday sale still needs a rate)
--          3. add USD->USD = 1.0 so every sale can use the same join
-- =====================================================================================
DROP TABLE IF EXISTS silver.fx_rate_daily;
CREATE TABLE silver.fx_rate_daily (
  rate_date          TEXT    NOT NULL,
  from_currency      TEXT    NOT NULL,
  rate_to_usd        REAL    NOT NULL,
  is_carried_forward INTEGER NOT NULL,      -- 1 when no rate was published that day
  PRIMARY KEY (rate_date, from_currency)
);

DROP TABLE IF EXISTS temp._published;
CREATE TEMP TABLE _pu

In [41]:
run_step("pipeline/silver/02_fx_rate_daily.sql")
q("SELECT * FROM silver.fx_rate_daily WHERE rate_date BETWEEN '2025-09-11' AND '2025-09-16' ORDER BY from_currency, rate_date")

ran pipeline/silver/02_fx_rate_daily.sql


,rate_date,from_currency,rate_to_usd,is_carried_forward
0,2025-09-11,CAD,0.7472,0
1,2025-09-12,CAD,0.7508,0
2,2025-09-13,CAD,0.7508,1
3,2025-09-14,CAD,0.7508,1
4,2025-09-15,CAD,0.7447,0
5,2025-09-16,CAD,0.7475,0
6,2025-09-11,USD,1.0,0
7,2025-09-12,USD,1.0,0
8,2025-09-13,USD,1.0,0
9,2025-09-14,USD,1.0,0


Saturday 13 and Sunday 14 carry Friday 12's rate and are flagged.

**Check:** every day up to the as-of date has a CAD rate (a missing rate would later quarantine real sales).

In [42]:
q('''SELECT COUNT(*) AS days_without_a_cad_rate FROM silver.fiscal_calendar c
     WHERE c.calendar_date BETWEEN (SELECT MIN(rate_date) FROM silver.fx_rate_daily) AND '${AS_OF_DATE}'
       AND NOT EXISTS (SELECT 1 FROM silver.fx_rate_daily f WHERE f.rate_date = c.calendar_date AND f.from_currency = 'CAD')''')

,days_without_a_cad_rate
0,0


In [43]:
q("SELECT from_currency, COUNT(*) AS days, SUM(is_carried_forward) AS carried_forward_days, MIN(rate_date) AS first_day, MAX(rate_date) AS last_day FROM silver.fx_rate_daily GROUP BY 1")

,from_currency,days,carried_forward_days,first_day,last_day
0,CAD,634,188,2025-01-02,2026-09-27
1,USD,634,0,2025-01-02,2026-09-27


## Silver 03 · `store_history` (a Slowly Changing Dimension, type 2)

**From** `bronze.store_master_raw` (4 snapshots) + the three code lookups · **Grain** one row per store **per version**
of its attributes.

**Theory: slowly changing dimensions.** Store, product and customer attributes change *slowly* (a remodel, a move to a
new region). There are standard ways to handle a change:
* **Type 1:** overwrite the old value. Simple, but history is lost: after Boston's remodel, all its old sales would look
  like Supercenter sales.
* **Type 2:** keep **one row per version**, each with `valid_from` / `valid_to` and an `is_current` flag. Past sales can
  be matched to the version valid on the day of sale. We do this.

**Transformation 1: decode codes** with LEFT JOINs to the lookup tables: `FMT_CD 'NM'` → `Neighborhood Market`,
`RGN_CD '02'` → `Northeast`, `ST_CD 'MA'` → `Massachusetts`, `CNTRY 'US'` → `United States`.

**Transformation 2: normalise ids and dates:** `01042` → `1042`; US-style `05/14/2016` → `2016-05-14` using the
`MM/dd/yyyy` format. (The finance calendar uses `dd/MM/yyyy`. Using the wrong format for either would silently swap
day and month for the first 12 days of every month, which is why each file's format is stated explicitly.)

**Transformation 3: read the snapshot date from the file name** (`store_master_20260201.csv` → `2026-02-01`). This is
why bronze keeps `_source_file`.

In [44]:
q('''SELECT s._source_file,
            try_to_date(substr(s._source_file, -12, 8), 'yyyyMMdd') AS snapshot_date,
            s.STR_NBR, ltrim(s.STR_NBR, '0') AS store_id,
            s.FMT_CD, f.FMT_NM AS store_format, s.RGN_CD, r.RGN_NM AS region, s.ST_CD, st.ST_NM AS state_province,
            s.OPEN_DT, try_to_date(s.OPEN_DT, 'MM/dd/yyyy') AS open_date, s.SQFT
     FROM bronze.store_master_raw s
     LEFT JOIN bronze.ref_store_format_raw f ON f.FMT_CD = s.FMT_CD
     LEFT JOIN bronze.ref_region_raw r ON r.RGN_CD = s.RGN_CD
     LEFT JOIN bronze.ref_state_province_raw st ON st.ST_CD = s.ST_CD
     WHERE s.STR_NBR = '01042' ORDER BY snapshot_date''')

,_source_file,snapshot_date,STR_NBR,store_id,FMT_CD,store_format,RGN_CD,region,ST_CD,state_province,OPEN_DT,open_date,SQFT
0,stores/store_master_20250201.csv,2025-02-01,01042,1042,NM,Neighborhood Market,02,Northeast,MA,Massachusetts,05/14/2016,2016-05-14,38000
1,stores/store_master_20250601.csv,2025-06-01,01042,1042,NM,Neighborhood Market,02,Northeast,MA,Massachusetts,05/14/2016,2016-05-14,38000
2,stores/store_master_20260201.csv,2026-02-01,01042,1042,SC,Supercenter,02,Northeast,MA,Massachusetts,05/14/2016,2016-05-14,52000
3,stores/store_master_20260601.csv,2026-06-01,01042,1042,SC,Supercenter,02,Northeast,MA,Massachusetts,05/14/2016,2016-05-14,52000


Boston appears in 4 snapshots but only **changed once** (format `NM` → `SC`, 38,000 → 52,000 sq ft on 1 Feb 2026).

**Transformation 4: change detection.** Concatenate all attributes into one string (`attr_hash`); `LAG()` compares each
snapshot with the previous one for the same store; snapshots where nothing changed are dropped.

**Transformation 5: validity ranges.** `LEAD()` looks at the next version's start date: `valid_to` is the day before it,
or `9999-12-31` ("still valid") for the latest version, which also gets `is_current = 1`. The first version is valid
from the store's opening date.

In [45]:
show_sql("pipeline/silver/03_store_history.sql")

-- =====================================================================================
-- SILVER 03 · store_history   (slowly changing dimension, type 2)
-- Source : bronze.store_master_raw (4 monthly snapshots) + three code lookups
-- Grain  : one row per store per VERSION of its attributes
-- Steps  : 1. decode codes: FMT_CD 'NM' -> 'Neighborhood Market', RGN_CD '02' -> 'Northeast',
--             ST_CD 'MA' -> 'Massachusetts', CNTRY 'US' -> 'United States'
--          2. normalise ids ('01042' -> '1042') and parse US-style MM/DD/YYYY dates
--          3. read the snapshot date from the file name (store_master_20260201.csv)
--          4. keep a snapshot only when something changed since the previous snapshot
--          5. give each version a valid_from / valid_to range and flag the current one
-- Why    : history answers "what format was Boston when it sold this?"; gold uses
--          the current version, silver keeps them all.
-- ===============================================

In [46]:
run_step("pipeline/silver/03_store_history.sql")
q('''SELECT store_id, store_name, store_format, state_province, region, open_date, close_date, selling_area_sqft,
            valid_from, valid_to, is_current
     FROM silver.store_history WHERE store_id IN ('1042', '2211') ORDER BY store_id, valid_from''')

ran pipeline/silver/03_store_history.sql


,store_id,store_name,store_format,state_province,region,open_date,close_date,selling_area_sqft,valid_from,valid_to,is_current
0,1042,FreshCart Boston Seaport,Neighborhood Market,Massachusetts,Northeast,2016-05-14,<NA>,38000,2016-05-14,2026-01-31,0
1,1042,FreshCart Boston Seaport,Supercenter,Massachusetts,Northeast,2016-05-14,<NA>,52000,2026-02-01,9999-12-31,1
2,2211,FreshCart Atlanta Buckhead,Express,Georgia,Southeast,2017-03-18,<NA>,8800,2017-03-18,2026-05-31,0
3,2211,FreshCart Atlanta Buckhead,Express,Georgia,Southeast,2017-03-18,2026-05-30,8800,2026-06-01,9999-12-31,1


Boston has two versions (the remodel); Buckhead has two (the closure date appeared). All other stores have one.
35 snapshot rows became this many versions:

In [47]:
display(q("SELECT store_id, COUNT(*) AS versions, SUM(is_current) AS current_versions FROM silver.store_history GROUP BY 1 ORDER BY 2 DESC, 1"))
print("snapshot rows in bronze:", n("SELECT COUNT(*) FROM bronze.store_master_raw"), "-> versions in silver:", n("SELECT COUNT(*) FROM silver.store_history"))

,store_id,versions,current_versions
0,1042,2,1
1,2211,2,1
2,1107,1,1
3,2210,1,1
4,2305,1,1
5,3050,1,1
6,317,1,1
7,322,1,1
8,9001,1,1


snapshot rows in bronze: 35 -> versions in silver: 11


**Check:** exactly one current version per store, and versions of the same store never overlap. With that, the question
*"what format was Boston when it made this sale?"* has exactly one answer:

In [48]:
q('''SELECT '2025-12-15' AS sale_date, store_format FROM silver.store_history
     WHERE store_id = '1042' AND '2025-12-15' BETWEEN valid_from AND valid_to
     UNION ALL
     SELECT '2026-09-14', store_format FROM silver.store_history
     WHERE store_id = '1042' AND '2026-09-14' BETWEEN valid_from AND valid_to''')

,sale_date,store_format
0,2025-12-15,Neighborhood Market
1,2026-09-14,Supercenter


## Silver 04 · `product` and `product_cost`

**From** `bronze.product_master_raw`, `bronze.merch_hierarchy_raw`, `bronze.cost_history_raw` ·
**Grain** `product`: one row per SKU; `product_cost`: one row per SKU per cost version.

**The trap this step prevents: fan-out.** The ERP keeps one row per SKU *per sales organisation*. Watch what happens if
you join sales to the raw master as-is: every sale of a product with two rows **matches twice** and sales double.

In [49]:
q('''SELECT 'sales lines for SKU 4471023' AS what, COUNT(*) AS lines, ROUND(SUM(CAST(EXT_AMT AS REAL)), 2) AS gross
     FROM bronze.pos_tlog_raw WHERE ltrim(ITM_ID, '0') = '4471023'
     UNION ALL
     SELECT 'same lines joined to the RAW product master', COUNT(*), ROUND(SUM(CAST(s.EXT_AMT AS REAL)), 2)
     FROM bronze.pos_tlog_raw s JOIN bronze.product_master_raw p ON ltrim(p.MATNR, '0') = ltrim(s.ITM_ID, '0')
     WHERE ltrim(s.ITM_ID, '0') = '4471023' ''')

,what,lines,gross
0,sales lines for SKU 4471023,1314,6473.19
1,same lines joined to the RAW product master,2628,12946.38


That's the classic **fan-out** (or "join explosion") bug, and it's invisible unless you know the grain.

**Transformation 1: normalise the SKU** (`ltrim(MATNR, '0')`) so POS (`000004471023`), online (`4471023`) and ERP ids match.

**Transformation 2: deduplicate with a business rule.** `ROW_NUMBER() OVER (PARTITION BY sku ORDER BY CASE VKORG WHEN
'US01' THEN 1 ELSE 2 END)` ranks the US record first, so we **prefer US01** (the master copy, which also records
discontinued items) and **fall back to CA01** for the two Canada-only items.

In [50]:
q('''SELECT ltrim(MATNR, '0') AS product_id, VKORG, MAKTX, STATUS,
            ROW_NUMBER() OVER (PARTITION BY ltrim(MATNR, '0') ORDER BY CASE VKORG WHEN 'US01' THEN 1 ELSE 2 END) AS rn
     FROM bronze.product_master_raw WHERE MATNR IN ('4471023', '4471024', '4492050') ORDER BY 1, rn''')

,product_id,VKORG,MAKTX,STATUS,rn
0,4471023,US01,Crunchy Sea Salt Chips 200g,A,1
1,4471023,CA01,CRUNCHY SEA SALT CHIPS 200G,A,2
2,4471024,US01,BBQ Kettle Chips 200g,D,1
3,4471024,CA01,BBQ KETTLE CHIPS 200G,A,2
4,4492050,CA01,MAPLE BAKED BEANS 398ML,A,1


`rn = 1` wins. Note `4471024` is **Discontinued** in US01 but `A` (active) in the Canadian copy. Picking the wrong copy
would make out-of-stock reports count a discontinued item.

**Transformation 3: decode the hierarchy** by joining `MATKL` to `merch_hierarchy_raw` → department / category /
subcategory. **Transformation 4: decode units** (`EA` → Each, `KG` → Kilogram) and status (`A` → Active, `D` →
Discontinued), and fix the upper-case Canadian names with `initcap()`.

**Transformation 5 (`product_cost`): effective dates → validity ranges** with `LEAD()` (exactly the toy example in A11).
A sale then picks up the cost **valid on the day it was sold**, which matters because costs change during the year.

In [51]:
show_sql("pipeline/silver/04_product.sql")

-- =====================================================================================
-- SILVER 04 · product  and  product_cost
-- Source : bronze.product_master_raw  (one row per SKU PER SALES ORG: US01 and CA01)
--          bronze.merch_hierarchy_raw (material group code -> names)
--          bronze.cost_history_raw    (standard cost with effective dates)
-- Grain  : product      -> one row per SKU
--          product_cost -> one row per SKU per cost version
-- Steps  : 1. normalise the SKU (no leading zeros) so POS, online and ERP ids match
--          2. DEDUPLICATE: prefer the US01 record, fall back to CA01 for Canada-only items.
--             Without this, every sale joins to two product rows and sales double.
--          3. decode MATKL 'C0412' -> Grocery / Snacks / Chips and Crisps
--          4. decode unit and status codes; fix the upper-case CA01 descriptions
--          5. turn cost effective dates into valid_from / valid_to ranges
-- ===================================

In [52]:
run_step("pipeline/silver/04_product.sql")
display(q("SELECT * FROM silver.product WHERE product_id IN ('4471023', '4471024', '4492050')"))
print("product master rows in bronze:", n("SELECT COUNT(*) FROM bronze.product_master_raw"),
      "-> products in silver:", n("SELECT COUNT(*) FROM silver.product"))

ran pipeline/silver/04_product.sql


,product_id,product_name,brand,department,category,subcategory,unit_of_measure,product_status,material_group_code,source_sales_org
0,4471023,Crunchy Sea Salt Chips 200g,Crunchwell,Grocery,Snacks,Chips and Crisps,Each,Active,C0412,US01
1,4471024,BBQ Kettle Chips 200g,Crunchwell,Grocery,Snacks,Chips and Crisps,Each,Discontinued,C0412,US01
2,4492050,Maple Baked Beans 398ml,Pantry Lane,Grocery,Pantry,Canned Goods,Each,Active,C0432,CA01


product master rows in bronze: 162 -> products in silver: 82


In [53]:
q("SELECT * FROM silver.product_cost WHERE product_id IN ('20101', '20201') ORDER BY product_id, valid_from")

,product_id,valid_from,valid_to,unit_cost_usd
0,20101,2024-01-01,2025-12-27,1.1
1,20101,2025-12-28,9999-12-31,1.14
2,20201,2024-01-01,2026-07-12,2.42
3,20201,2026-07-13,9999-12-31,2.53


**Checks:** one row per SKU; every product has a decoded department; cost ranges never overlap.

In [54]:
q('''SELECT (SELECT COUNT(*) - COUNT(DISTINCT product_id) FROM silver.product) AS duplicate_skus,
            (SELECT COUNT(*) FROM silver.product WHERE department IS NULL) AS undecoded_categories,
            (SELECT COUNT(*) FROM silver.product_cost a JOIN silver.product_cost b
               ON a.product_id = b.product_id AND a.valid_from < b.valid_from AND a.valid_to >= b.valid_from) AS overlapping_cost_ranges''')

,duplicate_skus,undecoded_categories,overlapping_cost_ranges
0,0,0,0


## Silver 05 · `promotion` and `promotion_item`

**From** `bronze.promo_calendar_raw`, `bronze.promo_item_raw` · **Grain** one row per promotion; one row per promotion per product.

A simple but necessary step:
1. **Decode the mechanic:** `BOGO` → Buy One Get One, `PCT` → Percent Off, `MULTI` → Multi-buy, `LOY` → Loyalty Price.
   Anything unexpected becomes `Other` instead of disappearing.
2. **Type** the dates and the discount %. `NULLIF(DISC_PCT, '')` turns the empty string into a real NULL, meaning
   "not applicable" (a BOGO has no percentage), which is different from 0%.
3. **Normalise product ids**, and `INSERT OR IGNORE` makes the item list unique per (promotion, product).

In [55]:
show_sql("pipeline/silver/05_promotion.sql")

-- =====================================================================================
-- SILVER 05 · promotion  and  promotion_item
-- Source : bronze.promo_calendar_raw, bronze.promo_item_raw
-- Grain  : promotion      -> one row per promotion
--          promotion_item -> one row per promotion per product
-- Steps  : 1. decode the mechanic: BOGO -> 'Buy One Get One', PCT -> 'Percent Off',
--             MULTI -> 'Multi-buy', LOY -> 'Loyalty Price'
--          2. type dates and the discount percentage
--          3. normalise product ids
-- =====================================================================================
DROP TABLE IF EXISTS silver.promotion;
CREATE TABLE silver.promotion (
  promotion_id       TEXT PRIMARY KEY,
  promotion_name     TEXT NOT NULL,
  promotion_mechanic TEXT NOT NULL,
  start_date         TEXT NOT NULL,
  end_date           TEXT NOT NULL,
  discount_pct       REAL
);

INSERT INTO silver.promotion
SELECT
  PROMO_CD,
  PROMO_DESC,
  CASE MECH_CD WH

In [56]:
run_step("pipeline/silver/05_promotion.sql")
examples = "('BG26W33', 'PC25W08', 'LY25W05', 'MB25W03')"
display(q(f"SELECT PROMO_CD, PROMO_DESC, MECH_CD, START_DT, END_DT, DISC_PCT FROM bronze.promo_calendar_raw WHERE PROMO_CD IN {examples}"))
display(q(f"SELECT * FROM silver.promotion WHERE promotion_id IN {examples}"))
q("SELECT promotion_mechanic, COUNT(*) AS promotions, COUNT(discount_pct) AS with_a_pct FROM silver.promotion GROUP BY 1")

ran pipeline/silver/05_promotion.sql


,PROMO_CD,PROMO_DESC,MECH_CD,START_DT,END_DT,DISC_PCT
0,MB25W03,Holiday Hosting - 3 for the price of 2,MULTI,2025-02-16,2025-03-01,
1,LY25W05,Snack Attack - Members save 15%,LOY,2025-03-02,2025-03-08,15
2,PC25W08,Fresh Deals - 25% off,PCT,2025-03-23,2025-04-05,25
3,BG26W33,Snack Attack - Buy one get one free,BOGO,2026-09-13,2026-09-19,


,promotion_id,promotion_name,promotion_mechanic,start_date,end_date,discount_pct
0,BG26W33,Snack Attack - Buy one get one free,Buy One Get One,2026-09-13,2026-09-19,<NA>
1,LY25W05,Snack Attack - Members save 15%,Loyalty Price,2025-03-02,2025-03-08,15
2,MB25W03,Holiday Hosting - 3 for the price of 2,Multi-buy,2025-02-16,2025-03-01,<NA>
3,PC25W08,Fresh Deals - 25% off,Percent Off,2025-03-23,2025-04-05,25


,promotion_mechanic,promotions,with_a_pct
0,Buy One Get One,27,0
1,Loyalty Price,24,24
2,Multi-buy,29,0
3,Percent Off,47,47


## Silver 06 · `loyalty_member` and `loyalty_member_restricted`

**From** `bronze.crm_member_raw` · **Grain** one row per loyalty member (the latest extract wins).

**Theory: pseudonymisation with a hash.** A cryptographic hash such as **SHA-256** turns any text into a fixed
64-character fingerprint. The same input always gives the same output, so ids still **join**; but you can't turn the
fingerprint back into the card number.

In [57]:
q("SELECT '6034118822' AS card_number, sha2('6034118822', 256) AS customer_id, sha2('6034118823', 256) AS one_digit_different")

,card_number,customer_id,one_digit_different
0,6034118822,71f6e2e99597137c77eb9cd9bb4d0a74c98cd40699279b9be3f6fd922747df4e,90e83cdc47752d26723bfb24c769d474143c52a4012aab598273f1d332d0ef84


POS and online sales apply the **same function** to the card number they carry, so a member's purchases link to their
tier without the card number ever leaving silver.

The transformations:
1. Keep the **latest extract** per card (`ROW_NUMBER ... ORDER BY _source_file DESC`), in case several extracts land.
2. `customer_id = sha2(CARD_NBR, 256)`.
3. Replace the date of birth with an **age band** (`45-54`) computed at the as-of date. Useful for analysis; not identifying.
4. Decode tier `G/S/B` → Gold/Silver/Bronze; normalise the home store id.
5. Put card number, email and date of birth in a **separate restricted table**. On Databricks it gets a column mask
   and grants; it's **never** used by gold.

In [58]:
show_sql("pipeline/silver/06_loyalty_member.sql")

-- =====================================================================================
-- SILVER 06 · loyalty_member  and  loyalty_member_restricted
-- Source : bronze.crm_member_raw  (card number, email and date of birth = personal data)
-- Grain  : one row per loyalty member (latest extract wins)
-- Steps  : 1. replace the card number with a pseudonymous customer_id = sha2(card, 256).
--             POS and online sales use the same function, so the ids still join.
--          2. derive an age band instead of keeping the date of birth
--          3. decode tier 'G' -> 'Gold'; normalise the home store id
--          4. put the personal data in a separate RESTRICTED table.
--             On Databricks that table gets a column mask; it never flows to gold.
-- =====================================================================================
DROP TABLE IF EXISTS silver.loyalty_member;
CREATE TABLE silver.loyalty_member (
  customer_id     TEXT PRIMARY KEY,
  loyalty_tier    TEXT NOT

In [59]:
run_step("pipeline/silver/06_loyalty_member.sql")
print("BEFORE (bronze):")
display(q("SELECT CARD_NBR, EMAIL, DOB, TIER, ENROLL_DT, HOME_STR FROM bronze.crm_member_raw WHERE CARD_NBR = '6034118822'"))
print("AFTER, general table (no personal data):")
display(q("SELECT * FROM silver.loyalty_member WHERE customer_id = sha2('6034118822', 256)"))
print("AFTER, restricted table (personal data, locked down):")
display(q("SELECT substr(customer_id, 1, 16) || '...' AS customer_id, card_number, email, date_of_birth FROM silver.loyalty_member_restricted WHERE customer_id = sha2('6034118822', 256)"))

ran pipeline/silver/06_loyalty_member.sql
BEFORE (bronze):


,CARD_NBR,EMAIL,DOB,TIER,ENROLL_DT,HOME_STR
0,6034118822,member00001@example.com,1972-11-15,G,2019-04-02,01042


AFTER, general table (no personal data):


,customer_id,loyalty_tier,enrollment_date,age_band,home_store_id,extract_date
0,71f6e2e99597137c77eb9cd9bb4d0a74c98cd40699279b9be3f6fd922747df4e,Gold,2019-04-02,45-54,1042,2026-09-06


AFTER, restricted table (personal data, locked down):


,customer_id,card_number,email,date_of_birth
0,71f6e2e99597137c...,6034118822,member00001@example.com,1972-11-15


In [60]:
q("SELECT loyalty_tier, age_band, COUNT(*) AS members FROM silver.loyalty_member GROUP BY 1, 2 ORDER BY 1, 2").pivot(index="age_band", columns="loyalty_tier", values="members")

loyalty_tier,Bronze,Gold,Silver
age_band,,,
18-24,33,6,23
25-34,136,42,81
35-44,124,48,81
45-54,109,33,92
55-64,113,36,88
65+,170,47,105


## Silver 07 · `pos_sales_line` and `quarantine_pos_sales_line`

**From** `bronze.pos_tlog_raw` + `silver.fx_rate_daily` · **Grain** one row per receipt line, **including** voids and
training transactions (flagged). This is the most important silver table: eight transformations. We'll look at each
on real rows before running the file.

**Before**, the golden receipt and a Canadian return:

In [61]:
q('''SELECT TRX_ID, STR_NBR, REG_NBR, LN_NBR, TRX_DT, TRX_TM, ITM_ID, QTY, EXT_AMT, DISC_AMT, TRX_TYP, LYL_CARD_NBR, PROMO_CD, CRNCY_CD
     FROM bronze.pos_tlog_raw WHERE TRX_ID IN ('88120431', '88120519') ORDER BY TRX_ID, CAST(LN_NBR AS INTEGER)''')

,TRX_ID,STR_NBR,REG_NBR,LN_NBR,TRX_DT,TRX_TM,ITM_ID,QTY,EXT_AMT,DISC_AMT,TRX_TYP,LYL_CARD_NBR,PROMO_CD,CRNCY_CD
0,88120431,01042,03,1,20260914,93015,000004471023,2,6.98,3.49,S,6034118822,BG26W33,USD
1,88120431,01042,03,2,20260914,93015,000000093321,1.254,1.87,0.00,S,6034118822,,USD
2,88120431,01042,03,3,20260914,93015,000004471023,1,3.49,0.00,V,6034118822,,USD
3,88120431,01042,03,4,20260914,93015,000000005012,1,3.99,0.00,S,6034118822,,USD
4,88120519,00317,01,1,20260914,94210,000000551020,1,18.99,0.00,R,,,CAD


**Steps 1–3 · Type, normalise, decode.** Dates/times become a timestamp (`printf('%06d', 93015)` = `093015` restores the
missing leading zero); ids lose their padding; `S/R/V` become words; register `99` becomes a flag.

In [62]:
q('''SELECT TRX_DT, TRX_TM, try_to_date(TRX_DT, 'yyyyMMdd') AS sales_date, printf('%06d', CAST(TRX_TM AS INTEGER)) AS hhmmss,
            STR_NBR, ltrim(STR_NBR, '0') AS store_id, ITM_ID, ltrim(ITM_ID, '0') AS product_id,
            TRX_TYP, CASE TRX_TYP WHEN 'S' THEN 'Sale' WHEN 'R' THEN 'Return' WHEN 'V' THEN 'Void' END AS line_type,
            REG_NBR, CASE WHEN REG_NBR = '99' THEN 1 ELSE 0 END AS is_test_transaction
     FROM bronze.pos_tlog_raw WHERE TRX_ID = '88120431' ''')

,TRX_DT,TRX_TM,sales_date,hhmmss,STR_NBR,store_id,ITM_ID,product_id,TRX_TYP,line_type,REG_NBR,is_test_transaction
0,20260914,93015,2026-09-14,093015,01042,1042,000004471023,4471023,S,Sale,03,0
1,20260914,93015,2026-09-14,093015,01042,1042,000000093321,93321,S,Sale,03,0
2,20260914,93015,2026-09-14,093015,01042,1042,000004471023,4471023,V,Void,03,0
3,20260914,93015,2026-09-14,093015,01042,1042,000000005012,5012,S,Sale,03,0


**Step 1 detail · strict date parsing, and a trap worth knowing.** SQLite's own `date()` "helpfully" rolls an impossible
date forward. A sale dated 30 February would silently move into March. The strict parser returns NULL instead, so the
row is caught:

In [63]:
q("SELECT date('2025-02-30') AS sqlite_date_function, try_to_date('20250230', 'yyyyMMdd') AS strict_try_to_date")

,sqlite_date_function,strict_try_to_date
0,2025-03-02,None


**Step 4 · Sign.** Returns arrive with **positive** amounts. If we just summed `EXT_AMT`, a return would *add* to sales.
Multiplying by `-1` for `R` makes the sign carry the meaning, so `SUM()` nets returns off automatically.

**Step 5 · Pseudonymise** the loyalty card → `sha2(card)`, or `ANONYMOUS` when no card was scanned.

**Step 6 · Convert currency.** LEFT JOIN `silver.fx_rate_daily` on (sales date, currency) and multiply. Thanks to the
USD→USD = 1.0 rows, US and Canadian lines use the same join.

In [64]:
q('''SELECT p.TRX_ID, p.STR_NBR, p.TRX_TYP, p.EXT_AMT AS raw_amount, p.CRNCY_CD,
            CASE WHEN p.TRX_TYP = 'R' THEN -1 ELSE 1 END * CAST(p.EXT_AMT AS REAL) AS signed_local,
            fx.rate_to_usd,
            ROUND(CASE WHEN p.TRX_TYP = 'R' THEN -1 ELSE 1 END * CAST(p.EXT_AMT AS REAL) * fx.rate_to_usd, 2) AS signed_usd,
            CASE WHEN NULLIF(TRIM(p.LYL_CARD_NBR), '') IS NULL THEN 'ANONYMOUS' ELSE substr(sha2(p.LYL_CARD_NBR, 256), 1, 12) || '...' END AS customer_id
     FROM bronze.pos_tlog_raw p
     LEFT JOIN silver.fx_rate_daily fx ON fx.rate_date = try_to_date(p.TRX_DT, 'yyyyMMdd') AND fx.from_currency = upper(p.CRNCY_CD)
     WHERE p.TRX_ID IN ('88120519') OR (p.TRX_ID = '88120431' AND p.LN_NBR = '1')''')

,TRX_ID,STR_NBR,TRX_TYP,raw_amount,CRNCY_CD,signed_local,rate_to_usd,signed_usd,customer_id
0,88120431,01042,S,6.98,USD,6.98,1.0,6.98,71f6e2e99597...
1,88120519,00317,R,18.99,CAD,-18.99,0.7336,-13.93,ANONYMOUS


**Step 7 · Deduplicate the re-delivered day.** On 10 June 2025 the store system sent the same rows twice. The natural
key of a POS line is (store, register, receipt, line, date). `ROW_NUMBER()` over that key keeps the first copy.

In [65]:
q('''SELECT TRX_DT, COUNT(*) AS rows_in_bronze,
            COUNT(DISTINCT STR_NBR || '|' || REG_NBR || '|' || TRX_ID || '|' || LN_NBR) AS distinct_lines
     FROM bronze.pos_tlog_raw WHERE TRX_DT IN ('20250609', '20250610', '20250611') GROUP BY TRX_DT''')

,TRX_DT,rows_in_bronze,distinct_lines
0,20250609,155,155
1,20250610,200,100
2,20250611,133,133


**Step 8 · Validate and quarantine.** Some rows break a **hard rule**: no store, impossible date, unknown transaction
type, no item, a quantity above 500, no FX rate. Dropping them silently would hide a source problem; keeping them would
corrupt sales. So they go to `silver.quarantine_pos_sales_line` **with a reason**, where someone can fix the source.
The `CASE` gives the *first* rule each row breaks.

(On Databricks the same rule is a pipeline *expectation*:
`CONSTRAINT passes_rules EXPECT (failure_reason IS NULL) ON VIOLATION DROP ROW`, which also logs drop counts.)

Now the real file. Its temp tables `_pos_typed` (steps 1–7) and `_pos_checked` (step 8) are the stages we just looked at:

In [66]:
show_sql("pipeline/silver/07_pos_sales_line.sql")

-- =====================================================================================
-- SILVER 07 · pos_sales_line  (+ quarantine_pos_sales_line)
-- Source : bronze.pos_tlog_raw   (store POS transaction log, one file per month)
-- Grain  : one row per receipt line, INCLUDING voids and training transactions
--          (silver keeps every valid source row; gold decides what counts as a sale)
-- Steps  : 1. TYPE      string dates/times -> DATE/TIMESTAMP, numbers -> REAL
--          2. NORMALISE store and item ids lose their leading zeros ('01042' -> '1042')
--          3. DECODE    TRX_TYP S/R/V -> Sale/Return/Void; REG_NBR 99 -> test transaction
--          4. SIGN      returns arrive with positive values; make them negative
--          5. PSEUDONYMISE loyalty card -> customer_id = sha2(card), or 'ANONYMOUS'
--          6. CONVERT   local currency -> USD with the daily rate from silver.fx_rate_daily
--          7. DEDUPLICATE the day the store system re-sent (same store/register/rec

In [67]:
run_step("pipeline/silver/07_pos_sales_line.sql")
q('''SELECT sales_line_id, store_id, sales_ts, product_id, substr(customer_id, 1, 10) || '...' AS customer_id,
            promotion_code, line_type, is_test_transaction, quantity_units, currency_code,
            gross_sales_amount_local, discount_amount_local, fx_rate_to_usd, gross_sales_amount_usd, discount_amount_usd
     FROM silver.pos_sales_line WHERE transaction_id IN ('88120431', '88120519') ORDER BY sales_line_id''')

ran pipeline/silver/07_pos_sales_line.sql


,sales_line_id,store_id,sales_ts,product_id,customer_id,promotion_code,line_type,is_test_transaction,quantity_units,currency_code,gross_sales_amount_local,discount_amount_local,fx_rate_to_usd,gross_sales_amount_usd,discount_amount_usd
0,POS-1042-88120431-1,1042,2026-09-14 09:30:15,4471023,71f6e2e995...,BG26W33,Sale,0,2.0,USD,6.98,3.49,1.0,6.98,3.49
1,POS-1042-88120431-2,1042,2026-09-14 09:30:15,93321,71f6e2e995...,<NA>,Sale,0,1.254,USD,1.87,0.0,1.0,1.87,0.0
2,POS-1042-88120431-3,1042,2026-09-14 09:30:15,4471023,71f6e2e995...,<NA>,Void,0,1.0,USD,3.49,0.0,1.0,3.49,0.0
3,POS-1042-88120431-4,1042,2026-09-14 09:30:15,5012,71f6e2e995...,<NA>,Sale,0,1.0,USD,3.99,0.0,1.0,3.99,0.0
4,POS-317-88120519-1,317,2026-09-14 09:42:10,551020,ANONYMOUS...,<NA>,Return,0,-1.0,CAD,-18.99,0.0,0.7336,-13.93,0.0


Read it against the "before": store `01042` → `1042`, item `000004471023` → `4471023`, `20260914`+`93015` →
`2026-09-14 09:30:15`, the Toronto return is now **negative** and in USD, the card became a hash.
**The void (line 3) is still here, flagged `Void`.** Silver keeps every real source event (voids are useful for
fraud and shrink analysis); gold will decide it's not a sale. The id `sales_line_id` = channel + store + receipt + line
is unique, and becomes the primary key.

The quarantine table:

In [68]:
display(q("SELECT failure_reason, COUNT(*) AS rows FROM silver.quarantine_pos_sales_line GROUP BY 1 ORDER BY 2 DESC"))
q("SELECT failure_reason, raw_str_nbr, transaction_id, line_number, raw_trx_dt, product_id, raw_qty, _source_file, _row_number FROM silver.quarantine_pos_sales_line ORDER BY failure_reason LIMIT 5")

,failure_reason,rows
0,missing store number,18
1,implausible quantity: 9999,9
2,invalid transaction date: 20250230,8
3,invalid transaction date: 20260230,4


,failure_reason,raw_str_nbr,transaction_id,line_number,raw_trx_dt,product_id,raw_qty,_source_file,_row_number
0,implausible quantity: 9999,00317,70015881,2,20251127,5103,9999,pos/pos_tlog_202511.csv,4661
1,implausible quantity: 9999,00317,70002831,6,20250401,4493002,9999,pos/pos_tlog_202504.csv,129
2,implausible quantity: 9999,00322,70030338,3,20260731,20202,9999,pos/pos_tlog_202607.csv,5254
3,implausible quantity: 9999,01042,70029932,1,20260725,20103,9999,pos/pos_tlog_202607.csv,4045
4,implausible quantity: 9999,01107,70029768,6,20260722,4473001,9999,pos/pos_tlog_202607.csv,3609


**Check: every bronze row is accounted for.** Bronze rows = silver rows + quarantined rows + duplicates removed.
Nothing disappears without an explanation. This is called **row-count reconciliation**.

In [69]:
b  = n("SELECT COUNT(*) FROM bronze.pos_tlog_raw")
s  = n("SELECT COUNT(*) FROM silver.pos_sales_line")
qu = n("SELECT COUNT(*) FROM silver.quarantine_pos_sales_line")
dup = n("SELECT COUNT(*) FROM temp._pos_typed WHERE copy_no > 1")
print(f"bronze {b:,} = silver {s:,} + quarantined {qu:,} + duplicates {dup:,}  ->  {'OK' if b == s + qu + dup else 'MISMATCH'}")
q("SELECT line_type, is_test_transaction, currency_code, COUNT(*) AS lines FROM silver.pos_sales_line GROUP BY 1, 2, 3 ORDER BY 1, 2, 3")

bronze 98,192 = silver 98,053 + quarantined 39 + duplicates 100  ->  OK


,line_type,is_test_transaction,currency_code,lines
0,Return,0,CAD,111
1,Return,0,USD,315
2,Return,1,USD,1
3,Sale,0,CAD,23947
4,Sale,0,USD,73085
5,Sale,1,CAD,69
6,Sale,1,USD,168
7,Void,0,CAD,79
8,Void,0,USD,278


## Silver 08 · `online_sales_line`

**From** `bronze.ecom_order_raw` + `silver.fx_rate_daily` · **Grain** one row per order line, with **exactly the same
columns as `pos_sales_line`**, so gold can simply stack the two channels (this is called *conforming* the two sources
to one schema; `CREATE TABLE ... AS SELECT * FROM silver.pos_sales_line WHERE 0` copies the schema with no rows).

**Transformation 1 · Flatten nested JSON.** One order holds a `lines` array. `json_each(value, '$.lines')` produces one
row per element: one order → many rows. (Databricks: `explode(from_json(...))`.)

In [70]:
order_id = n("SELECT json_extract(value, '$.orderId') FROM bronze.ecom_order_raw WHERE _source_file LIKE '%202609%' ORDER BY _row_number LIMIT 1")
q(f'''SELECT json_extract(o.value, '$.orderId') AS order_id, json_extract(o.value, '$.orderTs') AS order_ts,
             json_extract(l.value, '$.lineNo') AS line_no, json_extract(l.value, '$.sku') AS sku,
             json_extract(l.value, '$.qty') AS qty, json_extract(l.value, '$.lineTotal') AS line_total,
             json_extract(l.value, '$.discount') AS discount, json_extract(l.value, '$.promoCode') AS promo_code
      FROM bronze.ecom_order_raw o, json_each(o.value, '$.lines') l
      WHERE json_extract(o.value, '$.orderId') = '{order_id}' ''')

,order_id,order_ts,line_no,sku,qty,line_total,discount,promo_code
0,W1007923,2026-09-01T23:25:38-04:00,1,20103,1,4.99,1.5,PC26W31
1,W1007923,2026-09-01T23:25:38-04:00,2,4493002,1,3.99,0.0,<NA>
2,W1007923,2026-09-01T23:25:38-04:00,3,5013,1,3.89,0.0,<NA>
3,W1007923,2026-09-01T23:25:38-04:00,4,552001,1,4.49,0.0,<NA>
4,W1007923,2026-09-01T23:25:38-04:00,5,4491001,2,4.98,0.0,<NA>
5,W1007923,2026-09-01T23:25:38-04:00,6,5102,1,5.49,0.0,<NA>


**Transformation 2 · The business date comes from the LOCAL timestamp.** `2026-09-01T23:25:38-04:00` is 11:25 pm in
New Jersey on **1 September**. Converted to UTC it's already 2 September. The store, the customer and Finance all
consider it a 1 September sale, so we take the first 10 characters of the local timestamp and never convert to UTC first.
This many orders would have landed on the wrong day:

In [71]:
q('''SELECT json_extract(value, '$.orderTs') AS local_ts,
            substr(json_extract(value, '$.orderTs'), 1, 10) AS business_date_local,
            date(json_extract(value, '$.orderTs')) AS date_if_converted_to_utc
     FROM bronze.ecom_order_raw WHERE json_extract(value, '$.orderId') = 'W1007923' ''')

,local_ts,business_date_local,date_if_converted_to_utc
0,2026-09-01T23:25:38-04:00,2026-09-01,2026-09-02


In [72]:
print("orders that would move to the wrong day in UTC:",
      n("SELECT COUNT(*) FROM bronze.ecom_order_raw WHERE substr(json_extract(value, '$.orderTs'), 1, 10) <> date(json_extract(value, '$.orderTs'))"))

orders that would move to the wrong day in UTC: 2066


**Transformation 3 · Decode status:** `CANCELLED` → `line_type = 'Cancelled'` (kept and flagged, excluded in gold).
**Transformation 4 · Pseudonymise and convert** exactly like POS: `sha2(loyaltyId)` (the web and the till carry the same
card number, so the ids match), and the FX join.

In [73]:
show_sql("pipeline/silver/08_online_sales_line.sql")

-- =====================================================================================
-- SILVER 08 · online_sales_line
-- Source : bronze.ecom_order_raw   (one JSON document per order, lines nested inside)
-- Grain  : one row per order line - the SAME columns as silver.pos_sales_line,
--          so gold can simply stack the two channels
-- Steps  : 1. FLATTEN   explode the nested "lines" array: one order -> many rows
--          2. BUSINESS DATE from the LOCAL timestamp. '2026-09-01T23:25:38-04:00' is a
--             1 September sale; converting to UTC first would move it to 2 September.
--          3. DECODE    status CANCELLED -> line_type 'Cancelled' (excluded in gold)
--          4. PSEUDONYMISE and CONVERT exactly like POS
-- =====================================================================================
DROP TABLE IF EXISTS silver.online_sales_line;
CREATE TABLE silver.online_sales_line AS
SELECT * FROM silver.pos_sales_line WHERE 0;          -- identical schema to the

In [74]:
run_step("pipeline/silver/08_online_sales_line.sql")
display(q(f'''SELECT sales_line_id, transaction_id, line_number, store_id, sales_date, sales_ts, product_id, sales_channel, line_type,
                    quantity_units, currency_code, gross_sales_amount_local, discount_amount_local, gross_sales_amount_usd
             FROM silver.online_sales_line WHERE transaction_id = '{order_id}' '''))
print("orders in bronze:", n("SELECT COUNT(*) FROM bronze.ecom_order_raw"),
      "-> order lines in silver:", n("SELECT COUNT(*) FROM silver.online_sales_line"),
      "| lines inside the JSON:", n("SELECT COUNT(*) FROM bronze.ecom_order_raw o, json_each(o.value, '$.lines')"))
q("SELECT line_type, COUNT(*) AS lines FROM silver.online_sales_line GROUP BY 1")

ran pipeline/silver/08_online_sales_line.sql


,sales_line_id,transaction_id,line_number,store_id,sales_date,sales_ts,product_id,sales_channel,line_type,quantity_units,currency_code,gross_sales_amount_local,discount_amount_local,gross_sales_amount_usd
0,WEB-W1007923-1,W1007923,1,9001,2026-09-01,2026-09-01 23:25:38,20103,Online,Sale,1,USD,4.99,1.5,4.99
1,WEB-W1007923-2,W1007923,2,9001,2026-09-01,2026-09-01 23:25:38,4493002,Online,Sale,1,USD,3.99,0.0,3.99
2,WEB-W1007923-3,W1007923,3,9001,2026-09-01,2026-09-01 23:25:38,5013,Online,Sale,1,USD,3.89,0.0,3.89
3,WEB-W1007923-4,W1007923,4,9001,2026-09-01,2026-09-01 23:25:38,552001,Online,Sale,1,USD,4.49,0.0,4.49
4,WEB-W1007923-5,W1007923,5,9001,2026-09-01,2026-09-01 23:25:38,4491001,Online,Sale,2,USD,4.98,0.0,4.98
5,WEB-W1007923-6,W1007923,6,9001,2026-09-01,2026-09-01 23:25:38,5102,Online,Sale,1,USD,5.49,0.0,5.49


orders in bronze: 8288 -> order lines in silver: 49919 | lines inside the JSON: 49919


,line_type,lines
0,Cancelled,1511
1,Sale,48408


**Check:** the two sales tables have identical columns, in the same order, so `UNION ALL` in gold is safe.

In [75]:
pos_cols = [r[1] for r in con.execute("PRAGMA silver.table_info(pos_sales_line)")]
web_cols = [r[1] for r in con.execute("PRAGMA silver.table_info(online_sales_line)")]
print("identical schemas:", pos_cols == web_cols, f"({len(pos_cols)} columns)")

identical schemas: True (23 columns)


## Silver 09 · `inventory_daily`

**From** `bronze.inventory_raw` · **Grain** one row per store, product and day.

1. **Type and normalise** ids and dates (same functions as before; consistency is the point).
2. **Negative stock is physically impossible.** It comes from *phantom inventory*: stock that was stolen, damaged or
   miscounted without being recorded, so the system subtracts sales from a number that was already too high... then too
   low. We **clamp it to zero** (`MAX(0, qty)`, because you can't sell stock you don't have) **and flag it**
   (`had_negative_stock = 1`) so the stock team can investigate. Never "fix" data silently: fix it *and* record that you did.
3. **Keep the latest delivery** if a snapshot is re-sent (`ROW_NUMBER ... ORDER BY _ingested_at DESC`).

In [76]:
show_sql("pipeline/silver/09_inventory_daily.sql")

-- =====================================================================================
-- SILVER 09 · inventory_daily
-- Source : bronze.inventory_raw   (nightly stock snapshot per store and item)
-- Grain  : one row per store, product and day
-- Steps  : 1. type and normalise ids and dates
--          2. negative stock is physically impossible ("phantom inventory" from
--             unrecorded shrink); clamp it to zero and FLAG it so it can be investigated
--          3. keep the latest delivery if a snapshot is re-sent
-- =====================================================================================
DROP TABLE IF EXISTS silver.inventory_daily;
CREATE TABLE silver.inventory_daily (
  snapshot_date      TEXT    NOT NULL,
  store_id           TEXT    NOT NULL,
  product_id         TEXT    NOT NULL,
  on_hand_units      REAL    NOT NULL,
  on_order_units     REAL    NOT NULL,
  had_negative_stock INTEGER NOT NULL,
  _ingested_at       TEXT,
  PRIMARY KEY (snapshot_date, store_i

In [77]:
run_step("pipeline/silver/09_inventory_daily.sql")
q('''SELECT i.STR_NBR, i.ITM_ID, i.SNAP_DT, i.OH_QTY AS raw_on_hand, s.store_id, s.product_id, s.snapshot_date, s.on_hand_units, s.had_negative_stock
     FROM bronze.inventory_raw i
     JOIN silver.inventory_daily s ON s.store_id = ltrim(i.STR_NBR, '0') AND s.product_id = ltrim(i.ITM_ID, '0')
                                  AND s.snapshot_date = try_to_date(i.SNAP_DT, 'yyyyMMdd')
     WHERE CAST(i.OH_QTY AS REAL) < 0 LIMIT 4''')

ran pipeline/silver/09_inventory_daily.sql


,STR_NBR,ITM_ID,SNAP_DT,raw_on_hand,store_id,product_id,snapshot_date,on_hand_units,had_negative_stock
0,00322,000004471023,20260628,-1,322,4471023,2026-06-28,0,1
1,00322,000000005012,20260629,-1,322,5012,2026-06-29,0,1
2,01042,000000093325,20260716,-3.000,1042,93325,2026-07-16,0,1
3,01042,000000094102,20260715,-3,1042,94102,2026-07-15,0,1


In [78]:
q('''SELECT COUNT(*) AS rows, SUM(had_negative_stock) AS clamped_rows, MIN(snapshot_date) AS first_day, MAX(snapshot_date) AS last_day,
            COUNT(DISTINCT store_id) AS stores, COUNT(DISTINCT product_id) AS products, SUM(on_hand_units < 0) AS negative_left
     FROM silver.inventory_daily''')

,rows,clamped_rows,first_day,last_day,stores,products,negative_left
0,52052,86,2026-06-28,2026-09-26,8,79,0


## Silver summary

13 tables. Note the two that exist **only in silver**: `quarantine_pos_sales_line` (broken rows, kept for fixing at the
source) and `loyalty_member_restricted` (personal data). Neither ever flows to gold.

In [79]:
layer_tables("silver")

,silver table,rows
0,fiscal_calendar,6573
1,fx_rate_daily,1268
2,inventory_daily,52052
3,loyalty_member,1367
4,loyalty_member_restricted,1367
5,online_sales_line,49919
6,pos_sales_line,98053
7,product,82
8,product_cost,114
9,promotion,127


---
# Part G · Gold: the business star schema

Gold is what the business and Genie see. Every gold load does three kinds of work:

1. **Select and conform:** pick the business columns, drop technical ones, take the current version of each record.
2. **Enrich and derive:** look up values from other tables, calculate amounts once, precompute flags.
3. **Protect integrity:** one row per key; every fact row points to a real dimension row or an explicit Unknown member.

And three mechanics hold for every table:

* **Tables come from contracts.** Each gold table is declared in a YAML file in `contracts/gold/`: every column's
  type, nullability and **comment** (a plain-English description Genie reads), plus keys, clustering and tags. The same
  contract generates the local DDL, the Databricks DDL and the data dictionary, so they can't drift apart.
* **Loads are upserts** (`INSERT ... ON CONFLICT DO UPDATE` locally, `MERGE` on Databricks). Gold tables are **not**
  dropped and rebuilt, so re-running never duplicates and late corrections update rows in place.
* **Order matters:** dimensions first, facts last, because facts look up dimensions. Locally SQLite **enforces** every
  foreign key, so a fact row pointing at a missing dimension row stops the pipeline.

| Order | Gold table | Built from | Grain | Load |
|---|---|---|---|---|
| 1 | `dim_date` | `fiscal_calendar` | one row per day | upsert (flags move daily) |
| 2 | `dim_store` | `store_history` (current) + `fiscal_calendar` | one row per store + UNKNOWN | upsert |
| 3 | `dim_product` | `product` | one row per SKU + UNKNOWN | upsert |
| 4 | `dim_promotion` | `promotion` | one row per promotion + NO_PROMO | upsert |
| 5 | `dim_customer` | `loyalty_member` + customer ids in sales | one row per customer + ANONYMOUS | merge (keeps placeholders) |
| 6 | `fct_sales_line` | `pos_sales_line` ∪ `online_sales_line` + dims + `product_cost` | one row per receipt line | **incremental** by watermark |
| 7 | `fct_inventory_daily` | `inventory_daily` + dims + `product_cost` | one row per store, product, day | upsert by key |

### A contract, and the table it creates
Here is one contract (the store dimension), and the SQLite DDL generated from it:

In [80]:
print((ROOT / "contracts/gold/dim_store.yml").read_text())
print("---- generated SQLite DDL ----")
print(contracts.sqlite_ddl(contracts.load("dim_store")))

table: dim_store
comment: >-
  One row per FreshCart store or Dark Store with current attributes, plus an UNKNOWN member.
  Use to filter or group sales and stock by store, format, city, state, region or country,
  and for store lists such as openings. Attribute history is in silver.store_history.
primary_key: [store_id]
cluster_by: [region]
tags: {domain: retail_performance, grain: store}
columns:
  - {name: store_id, type: STRING, nullable: false, comment: "FreshCart store number without leading zeros, for example 1042. Join key to the facts."}
  - {name: store_name, type: STRING, nullable: false, comment: "Trading name, for example FreshCart Boston Seaport. Use when a question names a store."}
  - {name: store_format, type: STRING, nullable: false, comment: "One of: Supercenter, Neighborhood Market, Express, Dark Store. Dark Stores fulfil online orders only."}
  - {name: city, type: STRING, comment: "City where the store is located."}
  - {name: state_province, type: STRING, comment

`STRICT` makes SQLite enforce column types; `NOT NULL`, `PRIMARY KEY` and `FOREIGN KEY` are enforced too. On
Databricks the same contract becomes `CREATE TABLE ... COMMENT ... PRIMARY KEY ... RELY ... CLUSTER BY ...` plus tags
(see `databricks/gold/00_create_gold_tables.sql`). Create all seven empty gold tables now:

In [81]:
contracts.create_gold_tables(con)
layer_tables("gold")

,gold table,rows
0,_column_comments,76
1,_table_comments,7
2,dim_customer,0
3,dim_date,0
4,dim_product,0
5,dim_promotion,0
6,dim_store,0
7,fct_inventory_daily,0
8,fct_sales_line,0


In [82]:
print(contracts.sqlite_ddl(contracts.load("fct_sales_line")))

CREATE TABLE IF NOT EXISTS gold.fct_sales_line (
  sales_line_id TEXT NOT NULL,
  transaction_id TEXT NOT NULL,
  sales_date TEXT NOT NULL,
  store_id TEXT NOT NULL,
  product_id TEXT NOT NULL,
  customer_id TEXT NOT NULL,
  promotion_id TEXT NOT NULL,
  region TEXT NOT NULL,
  sales_channel TEXT NOT NULL,
  line_type TEXT NOT NULL,
  quantity_units REAL NOT NULL,
  gross_sales_amount_usd REAL NOT NULL,
  discount_amount_usd REAL NOT NULL,
  net_sales_amount_usd REAL NOT NULL,
  cost_of_goods_usd REAL NOT NULL,
  gross_margin_usd REAL NOT NULL,
  net_sales_amount_local REAL NOT NULL,
  currency_code TEXT NOT NULL,
  is_promo_sale INTEGER NOT NULL CHECK (is_promo_sale IN (0, 1)),
  is_loyalty_sale INTEGER NOT NULL CHECK (is_loyalty_sale IN (0, 1)),
  _source_ingested_at TEXT,
  PRIMARY KEY (sales_line_id),
  FOREIGN KEY (sales_date) REFERENCES dim_date (calendar_date),
  FOREIGN KEY (store_id) REFERENCES dim_store (store_id),
  FOREIGN KEY (product_id) REFERENCES dim_product (product_id

## Gold 01 · `dim_date`

**From** `silver.fiscal_calendar` · **Grain** one row per day · **Load** upsert every run.

**Why precompute flags?** "Last week" in a 4-5-4 business means *the last fully completed fiscal week, Sunday to
Saturday*, not "the last 7 days" and not an ISO week. "YTD vs last year" means *the same fiscal days* last year, not
the same calendar dates. An AI (or a new analyst) will get these wrong if it has to work them out from raw dates. So
the pipeline works them out **once**, relative to the as-of date, and stores them as true/false columns. A question
becomes a simple filter: `WHERE is_last_completed_fiscal_week`.

Steps: (1) find "yesterday" and the last completed fiscal week relative to the as-of date; (2) copy the fiscal
attributes; (3) compute five rolling flags: `is_last_completed_fiscal_week`, `is_last_4_completed_fiscal_weeks`,
`is_current_fiscal_quarter`, `is_fiscal_ytd`, `is_prior_fiscal_ytd`.

In [83]:
show_sql("pipeline/gold/01_load_dim_date.sql")

-- =====================================================================================
-- GOLD 01 · dim_date
-- Built from : silver.fiscal_calendar
-- Grain      : one row per calendar day
-- Load       : upsert every run (the rolling flags move every day)
-- Steps      : 1. find "yesterday" and the last fully completed fiscal week relative to
--                 the as-of date (${AS_OF_DATE}; current_date() in production)
--              2. copy the fiscal attributes
--              3. precompute the rolling flags so "last week" or "YTD vs last year"
--                 becomes a simple filter instead of date arithmetic Genie must guess
-- =====================================================================================
WITH cal AS (
  SELECT * FROM silver.fiscal_calendar
),
yesterday AS (                                                                   -- step 1
  SELECT fiscal_year AS cur_fy, fiscal_quarter AS cur_fq, fiscal_day_of_year AS cur_fdoy
  FROM cal
  WHERE calendar_d

In [84]:
run_step("pipeline/gold/01_load_dim_date.sql")
q('''SELECT calendar_date, day_of_week_name, fiscal_year, fiscal_period_name, fiscal_week, fiscal_day_of_year,
            is_last_completed_fiscal_week, is_last_4_completed_fiscal_weeks, is_fiscal_ytd
     FROM gold.dim_date WHERE is_last_completed_fiscal_week ORDER BY calendar_date''')

ran pipeline/gold/01_load_dim_date.sql


,calendar_date,day_of_week_name,fiscal_year,fiscal_period_name,fiscal_week,fiscal_day_of_year,is_last_completed_fiscal_week,is_last_4_completed_fiscal_weeks,is_fiscal_ytd
0,2026-09-20,Sunday,2026,P08 Sep,34,232,1,1,1
1,2026-09-21,Monday,2026,P08 Sep,34,233,1,1,1
2,2026-09-22,Tuesday,2026,P08 Sep,34,234,1,1,1
3,2026-09-23,Wednesday,2026,P08 Sep,34,235,1,1,1
4,2026-09-24,Thursday,2026,P08 Sep,34,236,1,1,1
5,2026-09-25,Friday,2026,P08 Sep,34,237,1,1,1
6,2026-09-26,Saturday,2026,P08 Sep,34,238,1,1,1


"Last week" is exactly Sunday 20 – Saturday 26 September 2026. And YTD vs prior YTD compares the **same number of
days, starting on the same weekday**:

In [85]:
q('''SELECT fiscal_year, MIN(calendar_date) AS first_day, MAX(calendar_date) AS last_day, COUNT(*) AS days,
            MIN(CASE WHEN fiscal_day_of_year = 1 THEN day_of_week_name END) AS starts_on
     FROM gold.dim_date WHERE is_fiscal_ytd OR is_prior_fiscal_ytd GROUP BY fiscal_year''')

,fiscal_year,first_day,last_day,days,starts_on
0,2025,2025-02-02,2025-09-27,238,Sunday
1,2026,2026-02-01,2026-09-26,238,Sunday


Both periods are 238 days long and start on a Sunday. The prior year's YTD ends on **27 September 2025**, not 26 September: same fiscal day, different calendar date. A naive "same dates last year" filter would compare a different mix of weekdays.

## Gold 02 · `dim_store`

**From** `silver.store_history` (current version) + `silver.fiscal_calendar` · **Grain** one row per store + UNKNOWN.

Steps: (1) take the **current** version of each store from the SCD2 history; (2) look up the fiscal year and day on
which the store opened; (3) apply the **comparable (like-for-like) rule, once, here**; (4) derive the status and the
current-year comparable flag; (5) add the **UNKNOWN** member.

**Theory: like-for-like (LFL).** If you opened 3 new stores this year, total sales grow even if every existing store is
declining. LFL growth compares only stores that traded for the whole of both periods. FreshCart's rule: a store is
comparable in fiscal year *N* if it traded **all of year N-1**. So:
* opened on **day 1** of a fiscal year → comparable from the next year;
* opened on **any other day** → comparable from the year after next (its first year was partial);
* **Dark Stores** → never (they only fulfil online orders).

Writing the rule in one place means Genie never has to know it; it just filters on a flag or calls a trusted function.

In [86]:
show_sql("pipeline/gold/02_load_dim_store.sql")

-- =====================================================================================
-- GOLD 02 · dim_store
-- Built from : silver.store_history (current version only) + silver.fiscal_calendar
-- Grain      : one row per store, plus UNKNOWN
-- Load       : upsert every run (status and comparable flag depend on the as-of date)
-- Steps      : 1. take the CURRENT version of each store from the SCD2 history
--              2. look up the fiscal year and fiscal day on which the store opened
--              3. comparable (like-for-like) rule, written once, here:
--                   a store is comparable in fiscal year N if it traded ALL of year N-1
--                   -> opened on day 1 of a fiscal year : comparable from that year + 1
--                   -> opened on any other day          : comparable from that year + 2
--                   -> Dark Stores are never comparable
--              4. status (Open/Closed) and the current-year comparable flag
--              5. add the UNKN

In [87]:
run_step("pipeline/gold/02_load_dim_store.sql")
q('''SELECT store_id, store_name, store_format, region, open_date, open_fiscal_year, close_date, store_status,
            comparable_from_fiscal_year, is_comparable_store_current_fy
     FROM gold.dim_store ORDER BY store_id''')

ran pipeline/gold/02_load_dim_store.sql


,store_id,store_name,store_format,region,open_date,open_fiscal_year,close_date,store_status,comparable_from_fiscal_year,is_comparable_store_current_fy
0,1042,FreshCart Boston Seaport,Supercenter,Northeast,2016-05-14,2016,<NA>,Open,2018,1
1,1107,FreshCart Hartford West,Express,Northeast,2019-10-05,2019,<NA>,Open,2021,1
2,2210,FreshCart Atlanta Midtown,Neighborhood Market,Southeast,2014-08-09,2014,<NA>,Open,2016,1
3,2211,FreshCart Atlanta Buckhead,Express,Southeast,2017-03-18,2017,2026-05-30,Closed,2019,0
4,2305,FreshCart Chicago Lincoln Park,Supercenter,Midwest,2012-04-21,2012,<NA>,Open,2014,1
5,3050,FreshCart Denver Highlands,Neighborhood Market,West,2024-09-07,2024,<NA>,Open,2026,1
6,317,FreshCart Toronto Liberty Village,Supercenter,Canada,2018-06-02,2018,<NA>,Open,2020,1
7,322,FreshCart Mississauga Square One,Express,Canada,2025-06-14,2025,<NA>,Open,2027,0
8,9001,FreshCart Newark Fulfilment Center,Dark Store,Northeast,2023-03-11,2023,<NA>,Open,<NA>,0
9,UNKNOWN,Unknown store,Unknown,Unknown,<NA>,<NA>,<NA>,Unknown,<NA>,0


Read the last columns against the story: **Denver** (3050) opened mid-FY2024 → comparable from FY2026. **Mississauga**
(322) opened in FY2025 → not until FY2027. **Buckhead** (2211) is closed → not comparable. The **Newark Dark Store**
(9001) never is. And the `UNKNOWN` row will catch any sale whose store can't be matched.

## Gold 03 · `dim_product`

**From** `silver.product` · **Grain** one row per SKU + UNKNOWN.

Steps: (1) keep only business columns: drop `material_group_code` and `source_sales_org`, which are useful to engineers
but just noise to a business user or an AI choosing columns; (2) derive `is_private_label` from the brand (FreshCart
and FreshCart Select are the retailer's own brands, a common analysis dimension); (3) add the **UNKNOWN** member for
items that were sold but are missing from the master (the scanned item `9999999`).

In [88]:
show_sql("pipeline/gold/03_load_dim_product.sql")

-- =====================================================================================
-- GOLD 03 · dim_product
-- Built from : silver.product (already one row per SKU with decoded hierarchy)
-- Grain      : one row per product, plus UNKNOWN
-- Load       : upsert every run
-- Steps      : 1. keep only business columns (drop material_group_code, source_sales_org)
--              2. derive is_private_label from the brand
--              3. add the UNKNOWN member for items sold but missing from the master
-- =====================================================================================
INSERT INTO gold.dim_product (
  product_id, product_name, brand, department, category, subcategory, unit_of_measure,
  is_private_label, product_status)
SELECT
  product_id, product_name, brand, department, category, subcategory, unit_of_measure,   -- step 1
  CASE WHEN brand IN ('FreshCart', 'FreshCart Select') THEN 1 ELSE 0 END,              -- step 2
  product_status
FROM silver.product
WHERE 

In [89]:
run_step("pipeline/gold/03_load_dim_product.sql")
display(q("SELECT * FROM gold.dim_product WHERE product_id IN ('4471023', '4471024', '93321', '4492050', 'UNKNOWN')"))
q("SELECT department, COUNT(*) AS products, SUM(is_private_label) AS private_label FROM gold.dim_product GROUP BY 1 ORDER BY 2 DESC")

ran pipeline/gold/03_load_dim_product.sql


,product_id,product_name,brand,department,category,subcategory,unit_of_measure,is_private_label,product_status
0,4471023,Crunchy Sea Salt Chips 200g,Crunchwell,Grocery,Snacks,Chips and Crisps,Each,0,Active
1,4471024,BBQ Kettle Chips 200g,Crunchwell,Grocery,Snacks,Chips and Crisps,Each,0,Discontinued
2,4492050,Maple Baked Beans 398ml,Pantry Lane,Grocery,Pantry,Canned Goods,Each,0,Active
3,93321,Bananas,Sunny Acres,Produce,Fresh Fruit,Bananas and Tropical,Kilogram,0,Active
4,UNKNOWN,Unknown product,<NA>,Unknown,Unknown,Unknown,Each,0,Unknown


,department,products,private_label
0,Grocery,28,9
1,Produce,17,5
2,Dairy and Eggs,10,6
3,Meat and Seafood,8,3
4,Household,8,4
5,Bakery,6,3
6,Deli,5,3
7,Unknown,1,0


## Gold 04 · `dim_promotion`

**From** `silver.promotion` · **Grain** one row per promotion + NO_PROMO.

Steps: (1) copy the decoded attributes; (2) add **`NO_PROMO`** so full-price lines also point at a real promotion row.
Then "sales by promotion mechanic" includes a "No Promotion" group instead of silently dropping full-price sales in a join.

In [90]:
show_sql("pipeline/gold/04_load_dim_promotion.sql")

-- =====================================================================================
-- GOLD 04 · dim_promotion
-- Built from : silver.promotion
-- Grain      : one row per promotion, plus NO_PROMO
-- Load       : upsert every run
-- Steps      : 1. copy the decoded promotion attributes
--              2. add NO_PROMO so full-price lines still have a promotion to join to
-- =====================================================================================
INSERT INTO gold.dim_promotion (promotion_id, promotion_name, promotion_mechanic, start_date, end_date)
SELECT promotion_id, promotion_name, promotion_mechanic, start_date, end_date
FROM silver.promotion
WHERE 1
ON CONFLICT (promotion_id) DO UPDATE SET
  promotion_name = excluded.promotion_name, promotion_mechanic = excluded.promotion_mechanic,
  start_date = excluded.start_date, end_date = excluded.end_date;

INSERT INTO gold.dim_promotion (promotion_id, promotion_name, promotion_mechanic)
VALUES ('NO_PROMO', 'No promotion', '

In [91]:
run_step("pipeline/gold/04_load_dim_promotion.sql")
q("SELECT * FROM gold.dim_promotion WHERE promotion_id IN ('NO_PROMO', 'BG26W33', 'PC25W08', 'LY25W05', 'MB25W03')")

ran pipeline/gold/04_load_dim_promotion.sql


,promotion_id,promotion_name,promotion_mechanic,start_date,end_date
0,BG26W33,Snack Attack - Buy one get one free,Buy One Get One,2026-09-13,2026-09-19
1,LY25W05,Snack Attack - Members save 15%,Loyalty Price,2025-03-02,2025-03-08
2,MB25W03,Holiday Hosting - 3 for the price of 2,Multi-buy,2025-02-16,2025-03-01
3,NO_PROMO,No promotion,No Promotion,<NA>,<NA>
4,PC25W08,Fresh Deals - 25% off,Percent Off,2025-03-23,2025-04-05


## Gold 05 · `dim_customer`

**From** `silver.loyalty_member` + the customer ids seen in silver sales · **Grain** one row per loyalty customer +
ANONYMOUS · **Load** merge (not rebuild).

Steps: (1) upsert members from the CRM with **no personal data** (pseudonymous id, tier, age band, home store);
(2) add **`ANONYMOUS`** for baskets without a card; (3) **late-arriving members**.

**Theory: late-arriving dimensions.** A card scanned today may not be in the CRM extract until next week. If the fact
load insisted that every customer already exists, those sales would fail or be dropped. Instead we insert a
**placeholder** row (`loyalty_tier = 'Unknown'`, `is_placeholder = 1`) for every customer id seen in sales but not yet
in the CRM. When the next CRM extract arrives, step 1's upsert fills in the details and sets `is_placeholder = 0`.
This is why `dim_customer` is **merged, not rebuilt**: a rebuild would delete placeholders that facts point to.

In [92]:
show_sql("pipeline/gold/05_load_dim_customer.sql")

-- =====================================================================================
-- GOLD 05 · dim_customer
-- Built from : silver.loyalty_member  +  customer ids seen in silver sales lines
-- Grain      : one row per loyalty customer, plus ANONYMOUS
-- Load       : MERGE / upsert (not a rebuild) because of step 3
-- Steps      : 1. upsert members from the CRM extract (no personal data: pseudonymous id,
--                 tier, age band, home store)
--              2. add ANONYMOUS for baskets without a loyalty card
--              3. LATE-ARRIVING MEMBERS: a card scanned today may not be in the CRM
--                 extract until next week. Insert a placeholder (tier 'Unknown') so the
--                 sales still load; step 1 fills in the details when the CRM row arrives.
-- =====================================================================================
INSERT INTO gold.dim_customer (customer_id, loyalty_tier, age_band, home_store_id, enrollment_date,   -- step 1
     

In [93]:
run_step("pipeline/gold/05_load_dim_customer.sql")
q("SELECT loyalty_tier, is_placeholder, COUNT(*) AS customers FROM gold.dim_customer GROUP BY 1, 2 ORDER BY 2, 3 DESC")

ran pipeline/gold/05_load_dim_customer.sql


,loyalty_tier,is_placeholder,customers
0,Bronze,0,685
1,Silver,0,470
2,Gold,0,212
3,Anonymous,0,1
4,Unknown,1,5


**Check: no personal data reached gold.** The only customer columns are these:

In [94]:
print([r[1] for r in con.execute("PRAGMA gold.table_info(dim_customer)")])

['customer_id', 'loyalty_tier', 'age_band', 'home_store_id', 'enrollment_date', 'is_placeholder']


## Gold 06 · `fct_sales_line` (the main fact)

**From** `silver.pos_sales_line` ∪ `silver.online_sales_line`, joined to `dim_store`, `dim_product`, `dim_promotion`
and `silver.product_cost` · **Grain** one row per receipt / order line · **Load** incremental with a watermark, upsert
by `sales_line_id`.

Six steps, in the order the SQL runs them:

**Step 1 · Stack the two channels:** `UNION ALL` of the two identical silver tables. (`UNION ALL` keeps every row;
plain `UNION` would also remove duplicate rows, which is slower and could wrongly merge two genuine identical lines.)

**Step 2 · Apply the business inclusion rules:** keep `Sale` and `Return`; drop `Void`, `Cancelled` and training
transactions. **This is the moment "every source event" becomes "what the business counts as sales".**

In [95]:
q('''SELECT sales_channel, line_type, is_test_transaction, COUNT(*) AS silver_rows,
            CASE WHEN line_type IN ('Sale', 'Return') AND is_test_transaction = 0 THEN 'kept' ELSE 'excluded' END AS in_gold
     FROM (SELECT * FROM silver.pos_sales_line UNION ALL SELECT * FROM silver.online_sales_line)
     GROUP BY 1, 2, 3 ORDER BY 1, 2, 3''')

,sales_channel,line_type,is_test_transaction,silver_rows,in_gold
0,In-store,Return,0,426,kept
1,In-store,Return,1,1,excluded
2,In-store,Sale,0,97032,kept
3,In-store,Sale,1,237,excluded
4,In-store,Void,0,357,excluded
5,Online,Cancelled,0,1511,excluded
6,Online,Sale,0,48408,kept


**Step 3 · Resolve every key** with LEFT JOINs; unmatched keys get their Unknown member (`UNKNOWN` store/product,
`NO_PROMO`). Customers always match because of Gold 05 step 3.

**Step 4 · Copy `region` from `dim_store`.** It exists only for **row-level security** on Databricks (a row filter
such as "Northeast managers see only Northeast rows" needs the column on the fact itself). It's tagged
`purpose = security_only`; analysis uses `dim_store.region`.

**Step 5 · Calculate the money once:**

| Column | Formula | Why here |
|---|---|---|
| `net_sales_amount_usd` | gross − discount | the company's single definition of "sales" (tax excluded) |
| `cost_of_goods_usd` | quantity × the standard cost **valid on the sale date** | needs a *range join*: `sales_date BETWEEN valid_from AND valid_to` |
| `gross_margin_usd` | net sales − cost of goods | stored per line, so it is **additive**: any grouping sums correctly |

**Step 6 · Flags:** `is_promo_sale` (a promotion matched) and `is_loyalty_sale` (not anonymous).

**Theory: the watermark (incremental loading).** The fact has 145k rows here, but in real life billions. Reprocessing
everything every night is slow and expensive. So the load only picks up silver lines whose source file landed
**after** the newest line already in gold: `WHERE _ingested_at > MAX(_source_ingested_at)`. That max value is the
**watermark**. Combined with the upsert on `sales_line_id`, re-runs never duplicate and late corrections update in place.

In [96]:
show_sql("pipeline/gold/06_load_fct_sales_line.sql")

-- =====================================================================================
-- GOLD 06 · fct_sales_line
-- Built from : silver.pos_sales_line UNION ALL silver.online_sales_line
--              joined to gold.dim_store, gold.dim_product, gold.dim_promotion
--              and silver.product_cost
-- Grain      : one row per receipt / order line  (same grain as silver: nothing is summed)
-- Load       : INCREMENTAL. Only lines whose source file landed after the newest line
--              already in gold are processed (a "watermark"), then upserted by key,
--              so re-running never duplicates.
-- Steps      : 1. stack the two channels (identical silver schemas)
--              2. apply the business inclusion rules: keep Sale and Return,
--                 drop Void, Cancelled and training (test) transactions
--              3. resolve every key to a real dimension row or its Unknown member
--              4. copy region from dim_store (used only by row-level securit

In [97]:
print("watermark before the first load:", repr(n("SELECT COALESCE(MAX(_source_ingested_at), '') FROM gold.fct_sales_line")))
run_step("pipeline/gold/06_load_fct_sales_line.sql")
print("rows loaded:", f'{n("SELECT COUNT(*) FROM gold.fct_sales_line"):,}')
print("watermark after the load       :", n("SELECT MAX(_source_ingested_at) FROM gold.fct_sales_line"))

watermark before the first load: ''


ran pipeline/gold/06_load_fct_sales_line.sql
rows loaded: 145,866
watermark after the load       : 2026-09-30T10:36:52+00:00


The golden receipt, before (silver) and after (gold). **The void is gone**, and every line now has net sales, cost and margin:

In [98]:
display(q("SELECT sales_line_id, line_type, quantity_units, gross_sales_amount_usd, discount_amount_usd, promotion_code FROM silver.pos_sales_line WHERE transaction_id = '88120431' ORDER BY sales_line_id"))
q('''SELECT sales_line_id, line_type, product_id, quantity_units, gross_sales_amount_usd, discount_amount_usd, net_sales_amount_usd,
            cost_of_goods_usd, gross_margin_usd, promotion_id, is_promo_sale, is_loyalty_sale, region
     FROM gold.fct_sales_line WHERE transaction_id = '88120431' ORDER BY sales_line_id''')

,sales_line_id,line_type,quantity_units,gross_sales_amount_usd,discount_amount_usd,promotion_code
0,POS-1042-88120431-1,Sale,2.0,6.98,3.49,BG26W33
1,POS-1042-88120431-2,Sale,1.254,1.87,0.0,<NA>
2,POS-1042-88120431-3,Void,1.0,3.49,0.0,<NA>
3,POS-1042-88120431-4,Sale,1.0,3.99,0.0,<NA>


,sales_line_id,line_type,product_id,quantity_units,gross_sales_amount_usd,discount_amount_usd,net_sales_amount_usd,cost_of_goods_usd,gross_margin_usd,promotion_id,is_promo_sale,is_loyalty_sale,region
0,POS-1042-88120431-1,Sale,4471023,2.0,6.98,3.49,3.49,4.46,-0.97,BG26W33,1,1,Northeast
1,POS-1042-88120431-2,Sale,93321,1.254,1.87,0.0,1.87,1.15,0.72,NO_PROMO,0,1,Northeast
2,POS-1042-88120431-4,Sale,5012,1.0,3.99,0.0,3.99,2.59,1.4,NO_PROMO,0,1,Northeast


Look at line 1: two bags of chips on buy-one-get-one. Net sales $3.49, cost $4.46, **margin negative**. That's real
retail economics, and exactly what a governed margin measure must report correctly.

Where did the lines that couldn't be matched go? Nowhere: they're in the Unknown members, with their money.

In [99]:
q('''SELECT 'product = UNKNOWN' AS unknown_member, COUNT(*) AS lines, ROUND(SUM(net_sales_amount_usd), 2) AS net_sales FROM gold.fct_sales_line WHERE product_id = 'UNKNOWN'
     UNION ALL SELECT 'store = UNKNOWN', COUNT(*), ROUND(SUM(net_sales_amount_usd), 2) FROM gold.fct_sales_line WHERE store_id = 'UNKNOWN'
     UNION ALL SELECT 'promotion = NO_PROMO', COUNT(*), ROUND(SUM(net_sales_amount_usd), 2) FROM gold.fct_sales_line WHERE promotion_id = 'NO_PROMO'
     UNION ALL SELECT 'customer = ANONYMOUS', COUNT(*), ROUND(SUM(net_sales_amount_usd), 2) FROM gold.fct_sales_line WHERE customer_id = 'ANONYMOUS' ''')

,unknown_member,lines,net_sales
0,product = UNKNOWN,14,90.73
1,store = UNKNOWN,0,<NA>
2,promotion = NO_PROMO,125793,1001420.13
3,customer = ANONYMOUS,86281,684222.4


Why is `store = UNKNOWN` empty? Because every POS row without a store number was already **quarantined** in Silver 07,
and every online order names a real fulfilment store. The `UNKNOWN` store row is a **safety net**: if tomorrow's file
contains a store that isn't in the store master yet, its sales still load (to `UNKNOWN`) instead of failing or vanishing.
The 14 `UNKNOWN` product lines are the scans of item `9999999`, which isn't in the product master: real money, kept.

### Proof: the incremental load is idempotent
Run the same fact load again. The watermark filters out everything already loaded, so nothing changes.

In [100]:
before_rows, before_sales = n("SELECT COUNT(*) FROM gold.fct_sales_line"), n("SELECT SUM(net_sales_amount_usd) FROM gold.fct_sales_line")
run_step("pipeline/gold/06_load_fct_sales_line.sql")
after_rows, after_sales = n("SELECT COUNT(*) FROM gold.fct_sales_line"), n("SELECT SUM(net_sales_amount_usd) FROM gold.fct_sales_line")
print(f"rows  {before_rows:,} -> {after_rows:,}")
print(f"sales {before_sales:,.2f} -> {after_sales:,.2f}")

ran pipeline/gold/06_load_fct_sales_line.sql
rows  145,866 -> 145,866
sales 1,153,932.81 -> 1,153,932.81


## Gold 07 · `fct_inventory_daily`

**From** `silver.inventory_daily` + `dim_store`, `dim_product`, `silver.product_cost` · **Grain** one row per store,
ranged product and day · **Load** upsert by (snapshot_date, store_id, product_id); re-sending a day replaces it.

Steps: (1) resolve keys (or UNKNOWN); (2) copy region for security; (3) **value the stock** at the standard cost valid
on the snapshot date (range join again); (4) define **out-of-stock** = an **active** product with no sellable stock.
Products a store doesn't sell never appear in its snapshot, so they can't be counted as out of stock by mistake,
and discontinued products running down to zero aren't "out of stock" either.

**Theory reminder: stock is semi-additive.** You can add stock across stores and products *on one day*, but never
across days. Seven nightly counts of 10 units is 10 units of stock, not 70. The semantic layer enforces this (Part H).

In [101]:
show_sql("pipeline/gold/07_load_fct_inventory_daily.sql")

-- =====================================================================================
-- GOLD 07 · fct_inventory_daily
-- Built from : silver.inventory_daily joined to gold.dim_store, gold.dim_product
--              and silver.product_cost
-- Grain      : one row per store, ranged product and day
-- Load       : upsert by (snapshot_date, store_id, product_id); re-sending a day replaces it
-- Steps      : 1. resolve keys to dimension rows (or UNKNOWN)
--              2. copy region for row-level security
--              3. value the stock at the standard cost valid on the snapshot date
--              4. out-of-stock = an ACTIVE product with no sellable stock.
--                 Products a store does not range never appear in the snapshot, so they
--                 can never be counted as out of stock by mistake.
-- =====================================================================================
INSERT INTO gold.fct_inventory_daily (snapshot_date, store_id, product_id, region,

In [102]:
run_step("pipeline/gold/07_load_fct_inventory_daily.sql")
q('''SELECT i.snapshot_date, s.store_name, p.product_name, p.product_status, i.on_hand_units, i.on_hand_value_usd, i.is_out_of_stock
     FROM gold.fct_inventory_daily i JOIN gold.dim_store s USING (store_id) JOIN gold.dim_product p USING (product_id)
     WHERE i.snapshot_date = '2026-09-26' AND s.store_id = '1042' ORDER BY i.is_out_of_stock DESC, p.product_name LIMIT 8''')

ran pipeline/gold/07_load_fct_inventory_daily.sql


,snapshot_date,store_name,product_name,product_status,on_hand_units,on_hand_value_usd,is_out_of_stock
0,2026-09-26,FreshCart Boston Seaport,Aged Cheddar 400g,Active,0.0,0.0,1
1,2026-09-26,FreshCart Boston Seaport,Bartlett Pears,Active,0.0,0.0,1
2,2026-09-26,FreshCart Boston Seaport,Ground Beef 85% 500g,Active,0.0,0.0,1
3,2026-09-26,FreshCart Boston Seaport,Spring Mix 142g,Active,0.0,0.0,1
4,2026-09-26,FreshCart Boston Seaport,2% Milk 4L,Active,17.0,43.01,0
5,2026-09-26,FreshCart Boston Seaport,Atlantic Salmon Fillet,Active,3.827,67.89,0
6,2026-09-26,FreshCart Boston Seaport,Baby Spinach 142g,Active,11.0,25.85,0
7,2026-09-26,FreshCart Boston Seaport,Bananas,Active,22.329,20.54,0


## Gold summary and integrity check

Every foreign key must resolve. SQLite checks this for us (`PRAGMA foreign_key_check` returns the violating rows; empty = all good):

In [103]:
display(layer_tables("gold"))
violations = con.execute("PRAGMA gold.foreign_key_check").fetchall()
print("foreign-key violations in gold:", len(violations))

,gold table,rows
0,_column_comments,76
1,_table_comments,7
2,dim_customer,1373
3,dim_date,6573
4,dim_product,83
5,dim_promotion,128
6,dim_store,10
7,fct_inventory_daily,52052
8,fct_sales_line,145866


foreign-key violations in gold: 0


The gold table and column comments (from the contracts) are stored in `gold._column_comments`, the local stand-in for
Unity Catalog `COMMENT`s. These plain-English descriptions are what Genie reads to pick the right column:

In [104]:
q("SELECT column_name, data_type, is_nullable, comment FROM gold._column_comments WHERE table_name = 'fct_sales_line' ORDER BY ordinal")

,column_name,data_type,is_nullable,comment
0,sales_line_id,STRING,0,"Unique id of the line: channel, store or order, receipt and line n..."
1,transaction_id,STRING,0,Receipt or online order id. COUNT(DISTINCT transaction_id) gives t...
2,sales_date,DATE,0,Business date of the sale in store local time. Join to dim_date.ca...
3,store_id,STRING,0,"Selling store; for online orders, the fulfilling store or Dark Store."
4,product_id,STRING,0,Product sold or returned. UNKNOWN if the item was not in the produ...
5,customer_id,STRING,0,"Pseudonymous loyalty customer id, or ANONYMOUS when no card was sc..."
6,promotion_id,STRING,0,"Promotion applied to the line, or NO_PROMO."
7,region,STRING,0,Copy of dim_store.region used only for row-level security. Use dim...
8,sales_channel,STRING,0,In-store or Online.
9,line_type,STRING,0,Sale or Return.


---
# Part H · The semantic layer: define every KPI once

Even with a perfect star schema, people (and AI) can still compute a KPI wrongly: average line percentages, sum stock
across days, count void lines as baskets. The **semantic layer** defines each measure **once**, with its business
meaning and synonyms, so every question uses the same definition.

On Databricks these are **Unity Catalog metric views** (YAML in `semantic/*.yaml`, deployed with
`CREATE VIEW ... WITH METRICS LANGUAGE YAML`). They're queried with `MEASURE()`:

```sql
SELECT region, MEASURE(net_sales) FROM freshcart.semantic.sales_metrics WHERE is_last_completed_fiscal_week GROUP BY ALL
```

SQLite has no metric views, so `freshcart/metrics.py` reads **the same YAML** and translates such a query to plain SQL.
Here are the sales measures:

In [105]:
import yaml
spec = yaml.safe_load((ROOT / "semantic/sales_metrics.yaml").read_text())
pd.DataFrame([(m["name"], m["expr"], m.get("comment", "")) for m in spec["measures"]], columns=["measure", "definition", "comment"])

,measure,definition,comment
0,net_sales,SUM(net_sales_amount_usd),"Sales after discounts and returns, excluding tax, USD. The default..."
1,gross_sales,SUM(gross_sales_amount_usd),"Sales at shelf price before discounts, after returns, excluding ta..."
2,discount_amount,SUM(discount_amount_usd),
3,units_sold,SUM(quantity_units),Net units; returns subtract. Weighed items count in kilograms.
4,transactions,COUNT(DISTINCT transaction_id) FILTER (WHERE line_type = 'Sale'),Distinct receipts or online orders with at least one sale line.
5,average_basket_value,"MEASURE(net_sales) / NULLIF(MEASURE(transactions), 0)",
6,units_per_transaction,"MEASURE(units_sold) / NULLIF(MEASURE(transactions), 0)",
7,gross_margin,SUM(gross_margin_usd),
8,gross_margin_pct,"MEASURE(gross_margin) / NULLIF(MEASURE(net_sales), 0)",Ratio of summed margin to summed net sales. Never average line-lev...
9,sales_before_returns,SUM(net_sales_amount_usd) FILTER (WHERE line_type = 'Sale'),


### Question 1: net sales and margin by region last week
The example query Genie is given, run through the local metric-view engine:

In [106]:
sql = (ROOT / "genie/example_queries/01_net_sales_and_margin_by_region_last_week.sql").read_text()
print(sql)
cols, rows = metrics.run(con, sql)
pd.DataFrame(rows, columns=cols)

-- Title (the way users ask): What were net sales and margin by region last week?
-- Usage guidance: "Last week" always means the last completed fiscal week, Sunday to Saturday.
SELECT region,
       MEASURE(net_sales)        AS net_sales,
       MEASURE(gross_margin)     AS gross_margin,
       MEASURE(gross_margin_pct) AS gross_margin_pct
FROM   freshcart.semantic.sales_metrics
WHERE  is_last_completed_fiscal_week
GROUP BY ALL
ORDER BY net_sales DESC;



,region,net_sales,gross_margin,gross_margin_pct
0,Northeast,5579.91,2003.78,0.359106
1,Canada,2706.50,1020.47,0.377044
2,Midwest,2543.06,911.63,0.358478
3,Southeast,1065.82,388.83,0.364818
4,West,939.99,328.42,0.349387


And the plain SQL it expands to, so you can see there's no magic: a join of the fact to its dimensions, a filter
on the precomputed date flag, and **ratios composed from sums**:

In [107]:
print(metrics.translate(con, sql))

WITH mv_rows AS (
SELECT
  source.sales_date AS "sales_date",
  dt.day_of_week_name AS "day_of_week_name",
  dt.fiscal_year AS "fiscal_year",
  dt.fiscal_quarter AS "fiscal_quarter",
  dt.fiscal_period AS "fiscal_period",
  dt.fiscal_period_name AS "fiscal_period_name",
  dt.fiscal_week AS "fiscal_week",
  dt.fiscal_week_start_date AS "fiscal_week_start_date",
  dt.is_last_completed_fiscal_week AS "is_last_completed_fiscal_week",
  dt.is_last_4_completed_fiscal_weeks AS "is_last_4_completed_fiscal_weeks",
  dt.is_current_fiscal_quarter AS "is_current_fiscal_quarter",
  dt.is_fiscal_ytd AS "is_fiscal_ytd",
  dt.is_prior_fiscal_ytd AS "is_prior_fiscal_ytd",
  store.store_name AS "store_name",
  store.store_format AS "store_format",
  store.region AS "region",
  store.country AS "country",
  store.is_comparable_store_current_fy AS "is_comparable_store",
  product.department AS "department",
  product.category AS "category",
  product.subcategory AS "subcategory",
  product.brand AS "brand

### Why "ratio of sums" matters, on real data
Margin % by department: the correct `SUM(margin) / SUM(net sales)` against the tempting-but-wrong average of line-level
percentages. The gap here is up to about 0.8 percentage points (Deli). Small, but exactly the kind of difference that
makes two reports disagree and destroys trust in the numbers.

In [108]:
q('''SELECT p.department,
            ROUND(SUM(f.gross_margin_usd) / SUM(f.net_sales_amount_usd), 4)                          AS right_margin_pct,
            ROUND(AVG(f.gross_margin_usd / NULLIF(f.net_sales_amount_usd, 0)), 4)                    AS wrong_avg_of_line_pcts
     FROM gold.fct_sales_line f JOIN gold.dim_product p USING (product_id)
     JOIN gold.dim_date d ON d.calendar_date = f.sales_date
     WHERE d.is_fiscal_ytd GROUP BY 1 ORDER BY 1''')

,department,right_margin_pct,wrong_avg_of_line_pcts
0,Bakery,0.4979,0.4951
1,Dairy and Eggs,0.2377,0.2425
2,Deli,0.4378,0.4294
3,Grocery,0.3259,0.3253
4,Household,0.304,0.3099
5,Meat and Seafood,0.2508,0.2547
6,Produce,0.3563,0.3569
7,Unknown,1.0,1.0


### Why "semi-additive" matters, on real data
Produce stock at the end of last week. Summing all 7 nightly snapshots overstates stock about 7x; the metric view takes
the **last day** of the period (`window: semiadditive: last` in `semantic/inventory_metrics.yaml`).

In [109]:
display(q('''SELECT ROUND(SUM(i.on_hand_units), 1) AS wrong_sum_over_7_days,
                    ROUND(SUM(i.on_hand_units) FILTER (WHERE i.snapshot_date = (SELECT MAX(calendar_date) FROM gold.dim_date WHERE is_last_completed_fiscal_week)), 1) AS right_last_day
             FROM gold.fct_inventory_daily i JOIN gold.dim_product p USING (product_id) JOIN gold.dim_date d ON d.calendar_date = i.snapshot_date
             WHERE d.is_last_completed_fiscal_week AND p.department = 'Produce' '''))
cols, rows = metrics.run(con, '''SELECT MEASURE(on_hand_units) AS stock_units, MEASURE(on_hand_value) AS stock_value_usd
                                 FROM freshcart.semantic.inventory_metrics WHERE is_last_completed_fiscal_week AND department = 'Produce' ''')
pd.DataFrame(rows, columns=cols)

,wrong_sum_over_7_days,right_last_day
0,14002.7,1903.1


,stock_units,stock_value_usd
0,1903.059,4240.61


### The trusted like-for-like function
Complex, high-stakes calculations get a **trusted function** instead of trusting an AI to write them. It uses the
comparable-store rule from `dim_store` and compares year-to-date with the **same fiscal days** last year:

In [110]:
print((ROOT / "semantic/fn_like_for_like_sales.local.sql").read_text())
cols, rows = metrics.run(con, "SELECT * FROM freshcart.semantic.fn_like_for_like_sales(2026)")
pd.DataFrame(rows, columns=cols)

-- Local (SQLite) body of the trusted function freshcart.semantic.fn_like_for_like_sales.
-- The Databricks version is databricks/semantic/03_fn_like_for_like_sales.sql.
-- Parameter :p_fiscal_year  - fiscal year to evaluate, for example 2026.
--
-- Like-for-like (comparable store) growth:
--   * only stores that are open and comparable in the evaluated year
--     (comparable_from_fiscal_year is computed once, in gold.dim_store)
--   * this year to date versus the SAME fiscal days last year
--     (fiscal_day_of_year lines up weekdays because fiscal years start on a Sunday)
WITH cutoff AS (
  SELECT MAX(d.fiscal_day_of_year) AS last_day
  FROM   gold.fct_sales_line f
  JOIN   gold.dim_date d ON f.sales_date = d.calendar_date
  WHERE  d.fiscal_year = :p_fiscal_year
),
comp AS (
  SELECT store_id, region
  FROM   gold.dim_store
  WHERE  comparable_from_fiscal_year <= :p_fiscal_year
    AND  store_status = 'Open'
),
agg AS (
  SELECT
    c.region,
    COUNT(DISTINCT c.store_id) AS compar

,region,comparable_stores,net_sales_ty,net_sales_ly,lfl_growth_pct
0,Canada,1,79792.37,79063.36,0.009221
1,Midwest,1,84177.52,83642.02,0.006402
2,Northeast,2,81529.03,62338.23,0.307850
3,Southeast,1,40586.99,40809.28,-0.005447
4,West,1,39419.21,38917.49,0.012892


Northeast stands out. Only two of its stores are comparable (Newark is a Dark Store), and one of them, Boston Seaport,
was remodelled into a bigger Supercenter at the start of FY2026. Breaking the same calculation down by store shows
where the growth comes from. This is the kind of follow-up question a governed layer makes easy to answer consistently:

In [111]:
q('''WITH cutoff AS (SELECT MAX(d.fiscal_day_of_year) AS last_day FROM gold.fct_sales_line f
                     JOIN gold.dim_date d ON d.calendar_date = f.sales_date WHERE d.fiscal_year = 2026)
     SELECT s.store_name, s.store_format,
            ROUND(SUM(CASE WHEN d.fiscal_year = 2026 THEN f.net_sales_amount_usd END), 2) AS net_sales_ty,
            ROUND(SUM(CASE WHEN d.fiscal_year = 2025 THEN f.net_sales_amount_usd END), 2) AS net_sales_ly,
            ROUND(SUM(CASE WHEN d.fiscal_year = 2026 THEN f.net_sales_amount_usd END)
                  / SUM(CASE WHEN d.fiscal_year = 2025 THEN f.net_sales_amount_usd END) - 1, 4) AS growth
     FROM gold.fct_sales_line f JOIN gold.dim_date d ON d.calendar_date = f.sales_date
     JOIN gold.dim_store s USING (store_id) CROSS JOIN cutoff
     WHERE s.region = 'Northeast' AND s.comparable_from_fiscal_year <= 2026 AND s.store_status = 'Open'
       AND d.fiscal_year IN (2025, 2026) AND d.fiscal_day_of_year <= cutoff.last_day
     GROUP BY 1, 2''')

,store_name,store_format,net_sales_ty,net_sales_ly,growth
0,FreshCart Boston Seaport,Supercenter,57509.28,38574.73,0.4909
1,FreshCart Hartford West,Express,24019.75,23763.5,0.0108


---
# Part I · Proving it is right

A pipeline is only trustworthy if it **proves** its output. Three kinds of proof:

## I1 · Data-quality rules (must all return 0)
`freshcart/quality.py` holds rules that must never be broken. In a real deployment a non-zero result fails the run.

In [112]:
rules = pd.DataFrame(quality.run_rules(con), columns=["layer", "rule", "violations"])
rules["result"] = rules["violations"].map(lambda v: "PASS" if v == 0 else "FAIL")
rules

,layer,rule,violations,result
0,gold,"No voided, cancelled or training lines in the sales fact",0,PASS
1,gold,"Returns are negative, sales are positive",0,PASS
2,gold,net_sales = gross_sales - discount on every line,0,PASS
3,gold,gross_margin = net_sales - cost_of_goods on every line,0,PASS
4,gold,Every sales date is in dim_date,0,PASS
5,gold,Exactly 7 days flagged as last completed fiscal week,0,PASS
6,gold,YTD and prior YTD cover the same number of days,0,PASS
7,gold,No stock row for an UNKNOWN store or product,0,PASS
8,silver,One current version per store in the SCD2 history,0,PASS
9,silver,An FX rate exists for every day up to the as-of date,0,PASS


## I2 · Findings: what the pipeline found and handled
These are **expected** to be non-zero on this deliberately messy data. Each is a problem that was fixed, not hidden.

In [113]:
pd.DataFrame(quality.run_findings(con), columns=["finding", "count"])

,finding,count
0,POS rows in bronze,98192
1,Duplicate POS rows removed (store re-sent 10 June 2025),100
2,POS rows quarantined (broke a hard rule),39
3,Voided POS lines excluded from gold,357
4,Training-register lines excluded from gold,238
5,Cancelled online lines excluded from gold,1511
6,Online orders whose date would shift if converted to UTC,2066
7,Product master rows (SKU x sales org) in bronze,162
8,Products after deduplication,82
9,Sales lines mapped to the UNKNOWN product,14


## I3 · Independent reconciliation: raw files → gold, without SQL
The strongest test: recompute total net sales **directly from the raw files** in plain Python, applying the business
rules independently (skip duplicates, bad rows, voids, training, cancellations; sign returns; convert CAD), and compare
with gold. If the two independent implementations agree to the cent, the SQL pipeline is doing what we think it does.

In [114]:
fx = {}
with open(C.RAW_DIR / "finance/fx_rates.csv", newline="") as fh:
    published = {date.fromisoformat(r["RATE_DT"]): float(r["RATE"]) for r in csv.DictReader(fh)}
d, last = min(published), None
while d <= C.AS_OF_DATE:                                   # forward-fill weekends and holidays
    last = published.get(d, last); fx[d] = last
    d = date.fromordinal(d.toordinal() + 1)
rate = lambda ccy, day: 1.0 if ccy == "USD" else fx[day]

expected, seen = 0.0, set()
for path in sorted((C.RAW_DIR / "pos").glob("*.csv")):
    with open(path, newline="") as fh:
        for r in csv.DictReader(fh):
            key = (r["STR_NBR"], r["REG_NBR"], r["TRX_ID"], r["LN_NBR"], r["TRX_DT"])
            if key in seen: continue                       # re-delivered row
            seen.add(key)
            try: day = date(int(r["TRX_DT"][:4]), int(r["TRX_DT"][4:6]), int(r["TRX_DT"][6:]))
            except ValueError: continue                    # impossible date
            if not r["STR_NBR"] or abs(float(r["QTY"])) > 500: continue   # quarantined
            if r["TRX_TYP"] == "V" or r["REG_NBR"] == "99": continue      # void / training
            sign, k = (-1 if r["TRX_TYP"] == "R" else 1), rate(r["CRNCY_CD"], day)
            expected += round(round(sign * float(r["EXT_AMT"]) * k, 2) - round(sign * float(r["DISC_AMT"]) * k, 2), 2)
for path in sorted((C.RAW_DIR / "ecommerce").glob("*.ndjson")):
    with open(path) as fh:
        for line in fh:
            o = json.loads(line)
            if o["status"] == "CANCELLED": continue
            k = rate(o["currency"], date.fromisoformat(o["orderTs"][:10]))
            for ln in o["lines"]:
                expected += round(round(ln["lineTotal"] * k, 2) - round(ln["discount"] * k, 2), 2)

actual = n("SELECT SUM(net_sales_amount_usd) FROM gold.fct_sales_line")
print(f"net sales recomputed from raw files : {expected:,.2f} USD")
print(f"net sales in gold.fct_sales_line    : {actual:,.2f} USD")
print("MATCH" if abs(actual - expected) < 0.05 else "MISMATCH")

net sales recomputed from raw files : 1,153,932.81 USD
net sales in gold.fct_sales_line    : 1,153,932.81 USD
MATCH


## I4 · One receipt, traced through every layer
Lineage means you can follow any number back to its source. Here is receipt `88120431` line 2 (1.254 kg of bananas, a weighed item) from
the raw file to the metric:

In [115]:
print("1. RAW FILE  data/raw/pos/pos_tlog_202609.csv")
with open(C.RAW_DIR / "pos/pos_tlog_202609.csv", encoding="utf-8") as fh:
    header = fh.readline().strip()
    raw_line = next(l.strip() for l in fh if l.startswith("88120431,01042,03,2,"))
print("   ", header); print("   ", raw_line)
print("\n2. BRONZE    bronze.pos_tlog_raw (same text + audit columns)")
display(q("SELECT TRX_ID, STR_NBR, LN_NBR, TRX_DT, TRX_TM, ITM_ID, QTY, EXT_AMT, TRX_TYP, _source_file, _row_number FROM bronze.pos_tlog_raw WHERE TRX_ID = '88120431' AND LN_NBR = '2'"))
print("3. SILVER    silver.pos_sales_line (typed, normalised, converted)")
display(q("SELECT sales_line_id, store_id, sales_ts, product_id, line_type, quantity_units, currency_code, gross_sales_amount_usd, _source_file FROM silver.pos_sales_line WHERE sales_line_id = 'POS-1042-88120431-2'"))
print("4. GOLD      gold.fct_sales_line joined to its dimensions")
display(q('''SELECT f.sales_line_id, d.fiscal_year, d.fiscal_week, s.store_name, s.region, p.product_name, p.department, f.quantity_units,
                    f.net_sales_amount_usd, f.cost_of_goods_usd, f.gross_margin_usd, f.promotion_id
             FROM gold.fct_sales_line f JOIN gold.dim_date d ON d.calendar_date = f.sales_date
             JOIN gold.dim_store s USING (store_id) JOIN gold.dim_product p USING (product_id)
             WHERE f.sales_line_id = 'POS-1042-88120431-2' '''))

1. RAW FILE  data/raw/pos/pos_tlog_202609.csv
    TRX_ID,STR_NBR,REG_NBR,LN_NBR,TRX_DT,TRX_TM,ITM_ID,QTY,UNIT_PRC,EXT_AMT,DISC_AMT,TAX_AMT,TRX_TYP,LYL_CARD_NBR,PROMO_CD,CRNCY_CD,TNDR_CD
    88120431,01042,03,2,20260914,93015,000000093321,1.254,1.49,1.87,0.00,0.00,S,6034118822,,USD,CC

2. BRONZE    bronze.pos_tlog_raw (same text + audit columns)


,TRX_ID,STR_NBR,LN_NBR,TRX_DT,TRX_TM,ITM_ID,QTY,EXT_AMT,TRX_TYP,_source_file,_row_number
0,88120431,01042,2,20260914,93015,000000093321,1.254,1.87,S,pos/pos_tlog_202609.csv,2162


3. SILVER    silver.pos_sales_line (typed, normalised, converted)


,sales_line_id,store_id,sales_ts,product_id,line_type,quantity_units,currency_code,gross_sales_amount_usd,_source_file
0,POS-1042-88120431-2,1042,2026-09-14 09:30:15,93321,Sale,1.254,USD,1.87,pos/pos_tlog_202609.csv


4. GOLD      gold.fct_sales_line joined to its dimensions


,sales_line_id,fiscal_year,fiscal_week,store_name,region,product_name,department,quantity_units,net_sales_amount_usd,cost_of_goods_usd,gross_margin_usd,promotion_id
0,POS-1042-88120431-2,2026,33,FreshCart Boston Seaport,Northeast,Bananas,Produce,1.254,1.87,1.15,0.72,NO_PROMO


---
# Part J · Cheat sheet

## Every table in one place

| Layer | Table | Grain (one row per...) | Built from | Key transformations |
|---|---|---|---|---|
| Bronze | `_ingestion_log` | loaded file | the loader | makes ingestion incremental and idempotent |
| Bronze | `pos_tlog_raw` | receipt line as delivered | `pos/pos_tlog_*.csv` | none (text + audit columns) |
| Bronze | `ecom_order_raw` | online order (JSON text) | `ecommerce/*.ndjson` | none |
| Bronze | `product_master_raw` | SKU × sales org | `erp/product_master.csv` | none |
| Bronze | `merch_hierarchy_raw` | material group code | `erp/merch_hierarchy.csv` | none |
| Bronze | `cost_history_raw` | SKU cost change | `erp/cost_history.csv` | none |
| Bronze | `store_master_raw` | store × snapshot | `stores/store_master_*.csv` | none |
| Bronze | `ref_region_raw`, `ref_store_format_raw`, `ref_state_province_raw` | code | `stores/ref_*.csv` | none |
| Bronze | `crm_member_raw` | loyalty card | `crm/crm_members_*.csv` | none |
| Bronze | `promo_calendar_raw`, `promo_item_raw` | promotion; promotion × item | `promotions/*.csv` | none |
| Bronze | `fiscal_calendar_raw` | day | `finance/fiscal_calendar.csv` | none |
| Bronze | `fx_rate_raw` | business day × currency | `finance/fx_rates.csv` | none |
| Bronze | `inventory_raw` | store × item × night | `inventory/*.csv` | none |
| Silver 01 | `fiscal_calendar` | day | fiscal_calendar_raw | strict DD/MM/YYYY parse, labels → numbers, dedup, fiscal day of year |
| Silver 02 | `fx_rate_daily` | day × currency | fx_rate_raw + fiscal_calendar | forward-fill weekends, flag, add USD = 1 |
| Silver 03 | `store_history` | store × version | store_master_raw + 3 lookups | decode, MM/DD/YYYY, snapshot date from file name, change detection, SCD2 ranges |
| Silver 04 | `product` | SKU | product_master_raw + merch_hierarchy_raw | normalise SKU, dedup preferring US01, decode hierarchy/units/status, initcap |
| Silver 04 | `product_cost` | SKU × cost version | cost_history_raw | effective dates → valid_from/valid_to |
| Silver 05 | `promotion` | promotion | promo_calendar_raw | decode mechanic, type dates, empty → NULL |
| Silver 05 | `promotion_item` | promotion × product | promo_item_raw | normalise ids, unique |
| Silver 06 | `loyalty_member` | member | crm_member_raw | sha2 id, age band, decode tier, latest extract |
| Silver 06 | `loyalty_member_restricted` | member | crm_member_raw | personal data isolated (never to gold) |
| Silver 07 | `pos_sales_line` | receipt line | pos_tlog_raw + fx_rate_daily | type, normalise, decode, sign returns, pseudonymise, FX, dedup, validate |
| Silver 07 | `quarantine_pos_sales_line` | rejected line | pos_tlog_raw | rows breaking hard rules, with reason |
| Silver 08 | `online_sales_line` | order line | ecom_order_raw + fx_rate_daily | flatten JSON, local business date, decode status, same schema as POS |
| Silver 09 | `inventory_daily` | store × product × day | inventory_raw | type, normalise, clamp negatives + flag, dedup |
| Gold 01 | `dim_date` | day | fiscal_calendar | rolling flags: last week, last 4 weeks, quarter, YTD, prior YTD |
| Gold 02 | `dim_store` | store (+UNKNOWN) | store_history + fiscal_calendar | current version, open fiscal year, comparable rule, status |
| Gold 03 | `dim_product` | product (+UNKNOWN) | product | business columns only, private label flag |
| Gold 04 | `dim_promotion` | promotion (+NO_PROMO) | promotion | copy + NO_PROMO member |
| Gold 05 | `dim_customer` | customer (+ANONYMOUS) | loyalty_member + sales ids | no PII, late-arriving placeholders, merge |
| Gold 06 | `fct_sales_line` | receipt/order line | both silver sales + dims + product_cost | union channels, inclusion rules, Unknown members, net/COGS/margin, flags, watermark |
| Gold 07 | `fct_inventory_daily` | store × product × day | inventory_daily + dims + product_cost | stock value, out-of-stock definition |

## Local demo ↔ Databricks

| Concept | This notebook (SQLite) | Databricks (`databricks/`) |
|---|---|---|
| Layers | `bronze.db`, `silver.db`, `gold.db` attached as schemas | `freshcart.bronze / .silver / .gold / .semantic` in Unity Catalog |
| Landing zone | `data/raw/` | Volume `/Volumes/freshcart/landing/raw/` |
| Incremental ingestion | `bronze._ingestion_log` | streaming tables with `read_files` (Auto Loader) |
| Silver | `pipeline/silver/*.sql` | Lakeflow Declarative Pipeline `databricks/pipelines/02_silver.sql` |
| Data-quality rules | quarantine table + `quality.py` | `CONSTRAINT ... EXPECT ... ON VIOLATION DROP ROW` |
| Gold DDL | generated from contracts, `STRICT`, enforced PK/FK | generated from the same contracts, `PRIMARY KEY ... RELY`, `COMMENT`, `CLUSTER BY`, tags |
| Gold loads | `INSERT ... ON CONFLICT DO UPDATE` | `MERGE INTO` |
| Metric views | `freshcart/metrics.py` reading `semantic/*.yaml` | `CREATE VIEW ... WITH METRICS LANGUAGE YAML` |
| Security | not simulated | row filters (the `region` column), column masks, grants |
| "Today" | pinned `AS_OF_DATE` | `current_date()` |

## Glossary

**Audit columns** `_source_file`, `_row_number`, `_ingested_at`: where and when a row came from ·
**Conform** give the same thing the same id/format everywhere · **CTE** a named step inside a query (`WITH x AS`) ·
**Deduplicate** keep one row per natural key · **Dimension** descriptive table (who/what/where/when) ·
**Fact** table of events/measurements with numbers · **Fan-out** a join that multiplies rows because the other side
isn't unique · **Forward fill** carry the last known value into gaps · **Grain** what one row means ·
**Idempotent** running twice = running once · **Incremental** only process what's new · **Lineage** the path from
source to report · **Medallion** bronze/silver/gold layering · **MERGE / upsert** insert or update by key ·
**Pseudonymise** replace an identifier with an irreversible but joinable token · **Quarantine** keep rejected rows
with the reason · **SCD2** keep a row per version with validity dates · **Semi-additive** summable across some
dimensions but not time · **Star schema** facts in the middle, dimensions around · **Unknown member** explicit
dimension row for unmatched keys · **Watermark** the high-water mark of what's already loaded ·
**Window function** a per-row calculation over related rows (`ROW_NUMBER`, `LAG`, `LEAD`).

## Try it yourself
1. Add a new month: copy `data/raw/pos/pos_tlog_202609.csv` to `pos_tlog_202610.csv`, re-run Part E and Silver 07, and
   watch the dedup and bronze log handle it (then delete the copy).
2. Change `AS_OF_DATE` in `freshcart/config.py` and re-run: see the `dim_date` flags and comparable stores move.
3. Break a rule on purpose (for example remove `AND l.is_test_transaction = 0` from Gold 06 in a copy) and see which
   data-quality check or reconciliation catches it.
4. Read the Databricks versions in `databricks/` side by side with the local SQL you ran here.